# Energy Analysis

Energy scale definition on 1-9 scale (roughly):
1 - Chillout/Ambient - reference track: Poa Alpina
2 - Deep House/Minimal - reference track: Icicle
3 - Deep House - reference track: Territory
4 - Progressive House - reference track: Brainwasher
5 - Vocal House/Pop - reference track: Parasite
6 - Tech House - reference track: At Night
7 - Techno - refrence track: Power Drive
8 - Psytrance/Tech Trance - reference track:
9 - Hardstyle/Dubstep
Generic descriptors
1 - ambient
2 - low-energy warm-up
3 - 
4 - steady groove / moderate club energy
5 -
6 - strong dancefloor propulsion
7 -
8 - peak-time
9 - extremely intense music

In [2]:
# Imports and shared preprocessing for energy analysis
from pathlib import Path
import csv
import math
import warnings

import numpy as np
import librosa
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from plotly.colors import qualitative as plotly_qual
from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.linear_model import TweedieRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import KFold, RepeatedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

MIX_METADATA_CANDIDATES = {
    'aries': [
        Path('music/aries-mix/aries_mix_tracks.csv'),
        Path('../music/aries-mix/aries_mix_tracks.csv'),
    ],
    'ara': [
        Path('music/ara-mix/ara_mix_tracks.csv'),
        Path('../music/ara-mix/ara_mix_tracks.csv'),
    ],
}


def first_existing_path(candidates: list[Path]) -> Path:
    for candidate in candidates:
        resolved = candidate.expanduser().resolve()
        if resolved.exists():
            return resolved
    tried = [str(c.expanduser().resolve()) for c in candidates]
    raise FileNotFoundError(f'Could not find any candidate path. Tried: {tried}')


def safe_float(value):
    try:
        if value is None:
            return None
        value_str = str(value).strip()
        if value_str == '':
            return None
        value_float = float(value_str)
        return value_float if np.isfinite(value_float) else None
    except Exception:
        return None


def resolve_track_audio_path(metadata_path: Path, mp3_name: str) -> Path:
    base_dir = metadata_path.parent.resolve()
    candidates = [
        base_dir / mp3_name,
        Path('music/aries-mix') / mp3_name,
        Path('../music/aries-mix') / mp3_name,
        Path('music/ara-mix') / mp3_name,
        Path('../music/ara-mix') / mp3_name,
    ]
    for candidate in candidates:
        resolved = candidate.expanduser().resolve()
        if resolved.exists():
            return resolved
    tried = [str(c.expanduser().resolve()) for c in candidates]
    raise FileNotFoundError(f'Could not resolve audio for {mp3_name}. Tried: {tried}')


def load_mix_metadata_rows(mix_metadata_candidates: dict[str, list[Path]]) -> tuple[list[dict], dict[str, Path]]:
    rows = []
    metadata_paths = {}
    for mix_name, candidates in mix_metadata_candidates.items():
        metadata_path = first_existing_path(candidates)
        metadata_paths[mix_name] = metadata_path
        with metadata_path.open('r', newline='') as f:
            reader = csv.DictReader(f)
            for row in reader:
                row2 = dict(row)
                row2['mix_name'] = mix_name
                row2['metadata_path'] = str(metadata_path)
                row2['energy_float'] = safe_float(row.get('energy'))
                row2['bpm_float'] = safe_float(row.get('bpm'))
                row2['genre_clean'] = (str(row.get('genre', '')).strip() or 'unknown').lower()
                rows.append(row2)
    return rows, metadata_paths


energy_analysis_rows, energy_analysis_metadata_paths = load_mix_metadata_rows(MIX_METADATA_CANDIDATES)

print(f'Loaded {len(energy_analysis_rows)} metadata rows from {len(energy_analysis_metadata_paths)} mixes:')
for mix_name, metadata_path in energy_analysis_metadata_paths.items():
    count = sum(1 for row in energy_analysis_rows if row.get('mix_name') == mix_name)
    print(f'  - {mix_name}: {count} rows ({metadata_path})')


Loaded 48 metadata rows from 2 mixes:
  - aries: 25 rows (/Users/josephdaher/Git Repositories/djprojectexploration/music/aries-mix/aries_mix_tracks.csv)
  - ara: 23 rows (/Users/josephdaher/Git Repositories/djprojectexploration/music/ara-mix/ara_mix_tracks.csv)


In [3]:
# Tempo vs labeled energy (Aries + Ara mixes), colored by genre
from pathlib import Path
import csv

import numpy as np
import plotly.graph_objects as go
from plotly.colors import qualitative as plotly_qual

MIX_METADATA_CANDIDATES = {
    'aries': [
        Path('music/aries-mix/aries_mix_tracks.csv'),
        Path('../music/aries-mix/aries_mix_tracks.csv'),
    ],
    'ara': [
        Path('music/ara-mix/ara_mix_tracks.csv'),
        Path('../music/ara-mix/ara_mix_tracks.csv'),
    ],
}


def _resolve_first_existing(candidates: list[Path]) -> Path:
    if 'first_existing_path' in globals():
        return first_existing_path(candidates)
    for c in candidates:
        r = c.expanduser().resolve()
        if r.exists():
            return r
    tried = [str(c.expanduser().resolve()) for c in candidates]
    raise FileNotFoundError(f'Could not find any candidate path. Tried: {tried}')


def _safe_float(x):
    try:
        if x is None:
            return None
        s = str(x).strip()
        if s == '':
            return None
        return float(s)
    except Exception:
        return None


rows = []
metadata_paths = {}
for mix_name, candidates in MIX_METADATA_CANDIDATES.items():
    meta_path = _resolve_first_existing(candidates)
    metadata_paths[mix_name] = meta_path
    with meta_path.open('r', newline='') as f:
        reader = csv.DictReader(f)
        for row in reader:
            bpm = _safe_float(row.get('bpm'))
            energy = _safe_float(row.get('energy'))
            if bpm is None or energy is None:
                continue
            if not np.isfinite(bpm) or bpm <= 0:
                continue
            rows.append({
                'mix_name': mix_name,
                'track_number': str(row.get('track_number', '')).strip(),
                'title': str(row.get('title', '')).strip(),
                'artists': str(row.get('artists', '')).strip(),
                'genre': (str(row.get('genre', '')).strip() or 'unknown').lower(),
                'bpm': float(bpm),
                'energy': float(energy),
            })

if len(rows) == 0:
    raise RuntimeError('No valid rows with bpm + energy found across Aries/Ara metadata.')

energy = np.array([r['energy'] for r in rows], dtype=np.float64)
bpm = np.array([r['bpm'] for r in rows], dtype=np.float64)
genres = np.array([r['genre'] for r in rows], dtype=object)
unique_energy = np.array(sorted(set(energy.tolist())), dtype=np.float64)
unique_genres = sorted(set(genres.tolist()))

palette = (
    plotly_qual.Plotly
    + plotly_qual.Safe
    + plotly_qual.Dark24
    + plotly_qual.Alphabet
)
genre_colors = {g: palette[i % len(palette)] for i, g in enumerate(unique_genres)}

rng = np.random.default_rng(42)
fig = go.Figure()

for genre in unique_genres:
    idx = np.where(genres == genre)[0]
    xj = energy[idx] + rng.uniform(-0.08, 0.08, size=idx.size)

    custom = np.array([
        [
            rows[i].get('genre', ''),
            rows[i].get('mix_name', ''),
            rows[i].get('track_number', ''),
            rows[i].get('title', ''),
            rows[i].get('artists', ''),
            rows[i].get('energy', np.nan),
            rows[i].get('bpm', np.nan),
        ]
        for i in idx
    ], dtype=object)

    fig.add_trace(
        go.Scatter(
            x=xj,
            y=bpm[idx],
            mode='markers',
            name=genre,
            legendgroup=f'genre::{genre}',
            marker=dict(color=genre_colors[genre], size=8, line=dict(color='black', width=0.45), opacity=0.85),
            customdata=custom,
            hovertemplate=(
                'genre=%{customdata[0]}<br>'
                'mix=%{customdata[1]}<br>'
                'track=%{customdata[2]}<br>'
                'title=%{customdata[3]}<br>'
                'artists=%{customdata[4]}<br>'
                'energy=%{customdata[5]:.1f}<br>'
                'bpm=%{customdata[6]:.2f}<extra></extra>'
            ),
        )
    )

mean_x = []
mean_y = []
for e in unique_energy:
    m = energy == e
    if np.any(m):
        mean_x.append(float(e))
        mean_y.append(float(np.mean(bpm[m])))

if mean_x:
    fig.add_trace(
        go.Scatter(
            x=np.asarray(mean_x, dtype=np.float64),
            y=np.asarray(mean_y, dtype=np.float64),
            mode='lines+markers',
            name='combined bpm mean',
            marker=dict(size=7),
            line=dict(color='black', width=2),
            hovertemplate='energy=%{x:.1f}<br>mean bpm=%{y:.2f}<extra></extra>',
        )
    )

fig.update_layout(
    title='Tempo (BPM Tag) vs Labeled Energy (Aries + Ara, Genre Colors)',
    width=980,
    height=520,
    template='plotly_white',
    legend=dict(title='Genre'),
    xaxis=dict(title='Labeled Energy Tag', tickmode='array', tickvals=unique_energy.tolist()),
    yaxis=dict(title='Tempo (BPM tag)'),
)

fig.show()

by_mix = {m: int(np.sum(np.array([r['mix_name'] for r in rows], dtype=object) == m)) for m in sorted(metadata_paths.keys())}
print(f'Plotted {len(rows)} tracks for tempo-vs-energy from mixes: {by_mix}')
for mix_name, meta in metadata_paths.items():
    print(f'  - {mix_name}: {meta}')
print('Tip: hover any point to inspect song details; click to pin the tooltip.')


Plotted 46 tracks for tempo-vs-energy from mixes: {'ara': 21, 'aries': 25}
  - aries: /Users/josephdaher/Git Repositories/djprojectexploration/music/aries-mix/aries_mix_tracks.csv
  - ara: /Users/josephdaher/Git Repositories/djprojectexploration/music/ara-mix/ara_mix_tracks.csv
Tip: hover any point to inspect song details; click to pin the tooltip.


In [4]:
# Spectral and groove feature set vs labeled energy (Aries + Ara, optimized + Plotly, genre-colored)
from pathlib import Path
import csv
import math

import numpy as np
import librosa
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from plotly.colors import qualitative as plotly_qual

MIX_METADATA_CANDIDATES = {
    'aries': [
        Path('music/aries-mix/aries_mix_tracks.csv'),
        Path('../music/aries-mix/aries_mix_tracks.csv'),
    ],
    'ara': [
        Path('music/ara-mix/ara_mix_tracks.csv'),
        Path('../music/ara-mix/ara_mix_tracks.csv'),
    ],
}

# Speed / quality knobs
FAST_ANALYSIS = False
ANALYSIS_SECONDS = 90.0 if FAST_ANALYSIS else None
ANALYSIS_SAMPLE_RATE = int(globals().get('SPECTRUM_SAMPLE_RATE', 22050))
ANALYSIS_N_FFT = 2048 if FAST_ANALYSIS else 4096
ANALYSIS_HOP = 1024 if FAST_ANALYSIS else 512
EPS = 1e-12

# Feature config
LOW_BAND_HZ = (20.0, 120.0)
BAND_FLUX_BANDS_HZ = {
    'low': (20.0, 250.0),
    'mid': (250.0, 2000.0),
    'high': (2000.0, 8000.0),
}
TILT_MIN_HZ = 30.0
TILT_MAX_HZ = 12000.0
SPECTRAL_ROLLOFF_PERCENT = 0.05
DYN_FRAME_SEC = 0.050
DYN_HOP_SEC = 0.025
PERIODICITY_MIN_BPM = 80.0
PERIODICITY_MAX_BPM = 170.0
PERIODICITY_AROUND_BPM_FRAC = 0.20
RHYTHM_ENTROPY_PHASE_BINS = 16

FEATURE_SPECS = [
    ('spectral_centroid_hz', 'Spectral Centroid vs Energy', 'Spectral Centroid (Hz)', '.0f'),
    ('spectral_tilt_db_per_oct', 'Spectral Tilt vs Energy', 'Spectral Tilt (dB / octave)', '.2f'),
    ('spectral_rolloff_05_hz', 'Spectral Rolloff (5%) vs Energy', 'Spectral Rolloff 5% (Hz)', '.0f'),
    ('low_band_energy_ratio', 'Low-Band Energy Ratio vs Energy', 'Low-Band Energy Ratio', '.3f'),
    ('low_band_periodicity', 'Low-Band Periodicity vs Energy', 'Low-Band Periodicity (norm. ACF peak)', '.3f'),
    ('dynamic_range_db', 'Dynamic Range vs Energy', 'Dynamic Range (dB)', '.2f'),
    ('crest_factor_db', 'Crest Factor vs Energy', 'Crest Factor (dB)', '.2f'),
    ('onset_density', 'Onset Density vs Energy', 'Onset Density (onsets / sec)', '.3f'),
    ('onset_magnitude', 'Onset Magnitude vs Energy', 'Onset Magnitude (mean peak strength)', '.3f'),
    ('rhythmic_entropy', 'Rhythmic Entropy vs Energy', 'Rhythmic Entropy (0-1)', '.3f'),
    ('band_flux_low', 'Band Flux (Low) vs Energy', 'Band Flux Low', '.3f'),
    ('band_flux_mid', 'Band Flux (Mid) vs Energy', 'Band Flux Mid', '.3f'),
    ('band_flux_high', 'Band Flux (High) vs Energy', 'Band Flux High', '.3f'),
]


def _resolve_first_existing(candidates: list[Path]) -> Path:
    if 'first_existing_path' in globals():
        return first_existing_path(candidates)
    for c in candidates:
        r = c.expanduser().resolve()
        if r.exists():
            return r
    tried = [str(c.expanduser().resolve()) for c in candidates]
    raise FileNotFoundError(f'Could not find any candidate path. Tried: {tried}')


def _safe_float(x):
    try:
        if x is None:
            return None
        s = str(x).strip()
        if s == '':
            return None
        return float(s)
    except Exception:
        return None


def _resolve_track_audio_path(metadata_path: Path, mp3_name: str) -> Path:
    base_dir = metadata_path.parent.resolve()
    candidates = [
        base_dir / mp3_name,
        Path('music/aries-mix') / mp3_name,
        Path('../music/aries-mix') / mp3_name,
        Path('music/ara-mix') / mp3_name,
        Path('../music/ara-mix') / mp3_name,
    ]
    for c in candidates:
        r = c.expanduser().resolve()
        if r.exists():
            return r
    tried = [str(c.expanduser().resolve()) for c in candidates]
    raise FileNotFoundError(f'Could not resolve audio for {mp3_name}. Tried: {tried}')


def _window_dynamic_range_db(audio: np.ndarray, sr: int, *, frame_sec: float, hop_sec: float, eps: float) -> float:
    frame_len = max(1, int(round(float(frame_sec) * int(sr))))
    hop_len = max(1, int(round(float(hop_sec) * int(sr))))
    if audio.size < frame_len:
        rms = np.sqrt(np.mean(np.square(audio.astype(np.float64))) + float(eps))
        db = np.array([20.0 * np.log10(rms + float(eps))], dtype=np.float64)
    else:
        framed = librosa.util.frame(audio, frame_length=frame_len, hop_length=hop_len).T
        rms = np.sqrt(np.mean(np.square(framed.astype(np.float64)), axis=1) + float(eps))
        db = 20.0 * np.log10(rms + float(eps))
    return float(np.percentile(db, 95.0) - np.percentile(db, 10.0))


def _compute_low_band_periodicity(
    low_band_frame_energy: np.ndarray,
    sr: int,
    hop_length: int,
    *,
    bpm_hint: float | None,
    min_bpm: float,
    max_bpm: float,
    around_bpm_frac: float,
    eps: float,
) -> float:
    x = np.asarray(low_band_frame_energy, dtype=np.float64).reshape(-1)
    if x.size < 8:
        return float('nan')

    x = x - float(np.mean(x))
    if float(np.std(x)) <= 0.0:
        return 0.0

    ac = librosa.autocorrelate(x, max_size=x.size - 1)
    if ac.size < 3 or not np.isfinite(ac[0]) or float(ac[0]) <= 0.0:
        return 0.0

    frames_per_sec = float(sr) / float(hop_length)
    lag_min = max(1, int(round((60.0 / float(max_bpm)) * frames_per_sec)))
    lag_max = max(lag_min + 1, int(round((60.0 / float(min_bpm)) * frames_per_sec)))

    if bpm_hint is not None and np.isfinite(bpm_hint) and bpm_hint > 0:
        lag0 = int(round((60.0 / float(bpm_hint)) * frames_per_sec))
        half = max(1, int(round(float(around_bpm_frac) * lag0)))
        lag_min = max(1, lag0 - half)
        lag_max = max(lag_min + 1, lag0 + half)

    lag_min = min(lag_min, ac.size - 1)
    lag_max = min(lag_max, ac.size - 1)
    if lag_max <= lag_min:
        return float('nan')

    peak = float(np.max(ac[lag_min:lag_max + 1]))
    periodicity = peak / (float(ac[0]) + float(eps))
    return float(np.clip(periodicity, 0.0, 1.0))


def _compute_rhythmic_entropy(
    onset_env: np.ndarray,
    sr: int,
    hop_length: int,
    *,
    bpm_hint: float | None,
    phase_bins: int,
    eps: float,
) -> float:
    env = np.asarray(onset_env, dtype=np.float64).reshape(-1)
    env = np.maximum(env, 0.0)
    if env.size == 0:
        return float('nan')

    total = float(np.sum(env))
    if total <= float(eps):
        return 0.0

    bpm_use = None
    if bpm_hint is not None and np.isfinite(bpm_hint) and bpm_hint > 0:
        bpm_use = float(bpm_hint)
    else:
        try:
            bpm_est = librosa.feature.tempo(onset_envelope=env.astype(np.float32), sr=sr, hop_length=hop_length)
            bpm_est = float(np.asarray(bpm_est).reshape(-1)[0]) if np.asarray(bpm_est).size else float('nan')
            if np.isfinite(bpm_est) and bpm_est > 0:
                bpm_use = bpm_est
        except Exception:
            bpm_use = None

    if bpm_use is not None:
        beat_period = 60.0 / float(bpm_use)
        times = librosa.frames_to_time(np.arange(env.size), sr=sr, hop_length=hop_length).astype(np.float64)
        phase = np.mod(times, beat_period) / max(beat_period, float(eps))
        bins = np.linspace(0.0, 1.0, int(phase_bins) + 1)
        hist, _ = np.histogram(phase, bins=bins, weights=env)
        p = hist.astype(np.float64)
    else:
        p = env.astype(np.float64)

    p_sum = float(np.sum(p))
    if p_sum <= float(eps):
        return 0.0
    p = p / p_sum
    p = p[p > 0]
    if p.size == 0:
        return 0.0

    h = -float(np.sum(p * np.log(p + float(eps))))
    h_max = math.log(float(max(2, p.size)))
    if h_max <= 0:
        return 0.0
    return float(np.clip(h / h_max, 0.0, 1.0))


def _compute_spectral_rolloff_from_power(freqs: np.ndarray, power: np.ndarray, *, roll_percent: float, eps: float) -> float:
    p = np.asarray(power, dtype=np.float64)
    f = np.asarray(freqs, dtype=np.float64).reshape(-1)
    if p.ndim != 2 or f.size != p.shape[0] or p.shape[1] == 0:
        return float('nan')

    cum = np.cumsum(p, axis=0)
    total = cum[-1, :]
    thresh = float(roll_percent) * total

    idx = np.argmax(cum >= thresh[None, :], axis=0)
    idx = np.clip(idx, 0, f.size - 1)
    roll = f[idx]
    if roll.size == 0:
        return float('nan')
    return float(np.nanmedian(roll))


def _compute_band_flux(stft_mag: np.ndarray, mask: np.ndarray) -> float:
    if stft_mag.shape[1] < 2 or not np.any(mask):
        return 0.0
    band = np.asarray(stft_mag[mask, :], dtype=np.float64)
    if band.size == 0:
        return 0.0
    diff = np.diff(band, axis=1)
    pos = np.maximum(diff, 0.0)
    flux_frames = np.mean(pos, axis=0)
    return float(np.mean(flux_frames)) if flux_frames.size else 0.0


def _compute_track_features(
    audio: np.ndarray,
    sr: int,
    *,
    n_fft: int,
    hop_length: int,
    low_band_hz: tuple[float, float],
    band_flux_bands_hz: dict[str, tuple[float, float]],
    tilt_min_hz: float,
    tilt_max_hz: float,
    spectral_rolloff_percent: float,
    dyn_frame_sec: float,
    dyn_hop_sec: float,
    periodicity_min_bpm: float,
    periodicity_max_bpm: float,
    periodicity_around_bpm_frac: float,
    rhythm_entropy_phase_bins: int,
    bpm_hint: float | None,
    eps: float,
) -> dict:
    stft_mag = np.abs(
        librosa.stft(audio.astype(np.float32), n_fft=int(n_fft), hop_length=int(hop_length), center=True)
    ).astype(np.float64)
    power = np.square(stft_mag) + float(eps)
    freqs = librosa.fft_frequencies(sr=sr, n_fft=int(n_fft)).astype(np.float64)

    centroid_frames = np.sum(freqs[:, None] * stft_mag, axis=0) / (np.sum(stft_mag, axis=0) + float(eps))
    spectral_centroid_hz = float(np.nanmedian(centroid_frames))

    spectral_rolloff_05_hz = _compute_spectral_rolloff_from_power(
        freqs,
        power,
        roll_percent=float(spectral_rolloff_percent),
        eps=eps,
    )

    tilt_mask = (freqs >= float(tilt_min_hz)) & (freqs <= float(tilt_max_hz))
    if np.count_nonzero(tilt_mask) >= 8:
        x = np.log2(freqs[tilt_mask])
        y = 10.0 * np.log10(power[tilt_mask, :] + float(eps))
        x_centered = x - float(np.mean(x))
        denom = float(np.sum(np.square(x_centered)))
        if denom > 0.0:
            y_centered = y - np.mean(y, axis=0, keepdims=True)
            slopes = np.sum(x_centered[:, None] * y_centered, axis=0) / denom
            spectral_tilt_db_per_oct = float(np.nanmedian(slopes))
        else:
            spectral_tilt_db_per_oct = float('nan')
    else:
        spectral_tilt_db_per_oct = float('nan')

    low_mask = (freqs >= float(low_band_hz[0])) & (freqs <= float(low_band_hz[1]))
    low_power = np.sum(power[low_mask, :], axis=0) if np.any(low_mask) else np.zeros(power.shape[1], dtype=np.float64)
    total_power = np.sum(power, axis=0)
    low_band_energy_ratio = float(np.mean(low_power) / (np.mean(total_power) + float(eps)))

    low_band_periodicity = _compute_low_band_periodicity(
        low_power,
        sr,
        hop_length,
        bpm_hint=bpm_hint,
        min_bpm=periodicity_min_bpm,
        max_bpm=periodicity_max_bpm,
        around_bpm_frac=periodicity_around_bpm_frac,
        eps=eps,
    )

    dynamic_range_db = _window_dynamic_range_db(audio, sr, frame_sec=dyn_frame_sec, hop_sec=dyn_hop_sec, eps=eps)

    peak = float(np.max(np.abs(audio))) if audio.size else 0.0
    rms = float(np.sqrt(np.mean(np.square(audio.astype(np.float64))) + float(eps)))
    crest_factor_db = float(20.0 * np.log10((peak + float(eps)) / (rms + float(eps))))

    onset_env = librosa.onset.onset_strength(y=audio.astype(np.float32), sr=int(sr), hop_length=int(hop_length)).astype(np.float64)
    onset_env = np.maximum(onset_env, 0.0)

    onset_frames = librosa.onset.onset_detect(
        onset_envelope=onset_env,
        sr=int(sr),
        hop_length=int(hop_length),
        units='frames',
        backtrack=False,
    )
    duration_sec = float(audio.size) / float(sr)
    onset_density = float(onset_frames.size) / max(duration_sec, float(eps))
    onset_magnitude = float(np.mean(onset_env[onset_frames])) if onset_frames.size else (float(np.mean(onset_env)) if onset_env.size else 0.0)

    rhythmic_entropy = _compute_rhythmic_entropy(
        onset_env,
        sr,
        hop_length,
        bpm_hint=bpm_hint,
        phase_bins=int(rhythm_entropy_phase_bins),
        eps=eps,
    )

    flux_masks = {
        band: (freqs >= float(rng[0])) & (freqs <= float(rng[1]))
        for band, rng in band_flux_bands_hz.items()
    }
    band_flux_low = _compute_band_flux(stft_mag, flux_masks.get('low', np.zeros_like(freqs, dtype=bool)))
    band_flux_mid = _compute_band_flux(stft_mag, flux_masks.get('mid', np.zeros_like(freqs, dtype=bool)))
    band_flux_high = _compute_band_flux(stft_mag, flux_masks.get('high', np.zeros_like(freqs, dtype=bool)))

    return {
        'spectral_centroid_hz': float(spectral_centroid_hz),
        'spectral_tilt_db_per_oct': float(spectral_tilt_db_per_oct),
        'spectral_rolloff_05_hz': float(spectral_rolloff_05_hz),
        'low_band_energy_ratio': float(low_band_energy_ratio),
        'low_band_periodicity': float(low_band_periodicity),
        'dynamic_range_db': float(dynamic_range_db),
        'crest_factor_db': float(crest_factor_db),
        'onset_density': float(onset_density),
        'onset_magnitude': float(onset_magnitude),
        'rhythmic_entropy': float(rhythmic_entropy),
        'band_flux_low': float(band_flux_low),
        'band_flux_mid': float(band_flux_mid),
        'band_flux_high': float(band_flux_high),
    }


if '_aries_ara_feature_metric_cache_v2' not in globals() or not isinstance(_aries_ara_feature_metric_cache_v2, dict):
    _aries_ara_feature_metric_cache_v2 = {}

all_rows = []
metadata_paths = {}
for mix_name, candidates in MIX_METADATA_CANDIDATES.items():
    metadata_path = _resolve_first_existing(candidates)
    metadata_paths[mix_name] = metadata_path
    with metadata_path.open('r', newline='') as f:
        reader = csv.DictReader(f)
        for row in reader:
            row2 = dict(row)
            row2['mix_name'] = mix_name
            row2['metadata_path'] = str(metadata_path)
            all_rows.append(row2)

metrics = []
failed = []
cache_hits = 0

for row in all_rows:
    mix_name = str(row.get('mix_name', '')).strip() or 'unknown'
    metadata_path = Path(str(row.get('metadata_path', '')).strip())

    mp3_name = str(row.get('mp3_name', '')).strip()
    title = str(row.get('title', '')).strip() or mp3_name
    artists = str(row.get('artists', '')).strip()
    genre = (str(row.get('genre', '')).strip() or 'unknown').lower()
    track_number = str(row.get('track_number', '')).strip()
    energy_val = _safe_float(row.get('energy'))
    bpm_hint = _safe_float(row.get('bpm'))

    if not mp3_name or energy_val is None:
        failed.append((mix_name, title or mp3_name or '<unknown>', 'missing mp3_name or energy'))
        continue

    try:
        track_path = _resolve_track_audio_path(metadata_path, mp3_name)
        cache_key = (
            mix_name,
            str(track_path),
            float(energy_val),
            None if bpm_hint is None else float(bpm_hint),
            genre,
            int(ANALYSIS_SAMPLE_RATE),
            int(ANALYSIS_N_FFT),
            int(ANALYSIS_HOP),
            None if ANALYSIS_SECONDS is None else float(ANALYSIS_SECONDS),
            tuple(float(x) for x in LOW_BAND_HZ),
            tuple((k, float(v[0]), float(v[1])) for k, v in sorted(BAND_FLUX_BANDS_HZ.items())),
            float(TILT_MIN_HZ),
            float(TILT_MAX_HZ),
            float(SPECTRAL_ROLLOFF_PERCENT),
            float(DYN_FRAME_SEC),
            float(DYN_HOP_SEC),
            float(PERIODICITY_MIN_BPM),
            float(PERIODICITY_MAX_BPM),
            float(PERIODICITY_AROUND_BPM_FRAC),
            int(RHYTHM_ENTROPY_PHASE_BINS),
        )

        if cache_key in _aries_ara_feature_metric_cache_v2:
            entry = dict(_aries_ara_feature_metric_cache_v2[cache_key])
            entry['title'] = title
            entry['artists'] = artists
            entry['genre'] = genre
            entry['track_number'] = track_number
            entry['mp3_name'] = mp3_name
            metrics.append(entry)
            cache_hits += 1
            continue

        audio, sr = librosa.load(
            str(track_path),
            sr=ANALYSIS_SAMPLE_RATE,
            mono=True,
            duration=ANALYSIS_SECONDS,
        )
        audio = np.asarray(audio, dtype=np.float32)
        if audio.size < ANALYSIS_N_FFT:
            failed.append((mix_name, title, f'audio too short ({audio.size} samples)'))
            continue

        feat = _compute_track_features(
            audio,
            sr,
            n_fft=ANALYSIS_N_FFT,
            hop_length=ANALYSIS_HOP,
            low_band_hz=LOW_BAND_HZ,
            band_flux_bands_hz=BAND_FLUX_BANDS_HZ,
            tilt_min_hz=TILT_MIN_HZ,
            tilt_max_hz=TILT_MAX_HZ,
            spectral_rolloff_percent=SPECTRAL_ROLLOFF_PERCENT,
            dyn_frame_sec=DYN_FRAME_SEC,
            dyn_hop_sec=DYN_HOP_SEC,
            periodicity_min_bpm=PERIODICITY_MIN_BPM,
            periodicity_max_bpm=PERIODICITY_MAX_BPM,
            periodicity_around_bpm_frac=PERIODICITY_AROUND_BPM_FRAC,
            rhythm_entropy_phase_bins=RHYTHM_ENTROPY_PHASE_BINS,
            bpm_hint=bpm_hint,
            eps=EPS,
        )

        entry = {
            'mix_name': mix_name,
            'track_number': track_number,
            'title': title,
            'artists': artists,
            'genre': genre,
            'mp3_name': mp3_name,
            'bpm': float(bpm_hint) if bpm_hint is not None else float('nan'),
            'energy': float(energy_val),
            **feat,
        }
        metrics.append(entry)
        _aries_ara_feature_metric_cache_v2[cache_key] = dict(entry)

    except Exception as exc:
        failed.append((mix_name, title, str(exc)))

if len(metrics) == 0:
    raise RuntimeError('No valid tracks available for feature vs energy plotting.')

# Stable snapshot for downstream modeling cells.
simple_feature_metrics = [dict(m) for m in metrics]
simple_feature_keys = [spec[0] for spec in FEATURE_SPECS]

mix_names = np.array([m['mix_name'] for m in metrics], dtype=object)
genres = np.array([m.get('genre', 'unknown') or 'unknown' for m in metrics], dtype=object)
energy = np.array([m['energy'] for m in metrics], dtype=np.float64)
unique_energy = np.array(sorted(set(energy.tolist())), dtype=np.float64)
unique_genres = sorted(set(genres.tolist()))

palette = (
    plotly_qual.Plotly
    + plotly_qual.Safe
    + plotly_qual.Dark24
    + plotly_qual.Alphabet
)
genre_colors = {g: palette[i % len(palette)] for i, g in enumerate(unique_genres)}

cols = 2
rows = int(math.ceil(len(FEATURE_SPECS) / float(cols)))
fig = make_subplots(
    rows=rows,
    cols=cols,
    subplot_titles=[spec[1] for spec in FEATURE_SPECS],
    horizontal_spacing=0.10,
    vertical_spacing=0.08,
)

rng = np.random.default_rng(42)

for f_idx, (feature_key, _title, y_label, value_fmt) in enumerate(FEATURE_SPECS):
    row_i = (f_idx // cols) + 1
    col_i = (f_idx % cols) + 1

    values = np.array([m.get(feature_key, np.nan) for m in metrics], dtype=np.float64)

    for genre in unique_genres:
        idx = np.where(genres == genre)[0]
        idx = idx[np.isfinite(values[idx])]
        if idx.size == 0:
            continue

        xj = energy[idx] + rng.uniform(-0.08, 0.08, size=idx.size)

        custom = np.array([
            [
                metrics[i].get('genre', ''),
                metrics[i].get('mix_name', ''),
                metrics[i].get('track_number', ''),
                metrics[i].get('title', ''),
                metrics[i].get('artists', ''),
                metrics[i].get('energy', np.nan),
                metrics[i].get('bpm', np.nan),
                metrics[i].get('spectral_centroid_hz', np.nan),
                metrics[i].get('spectral_tilt_db_per_oct', np.nan),
                metrics[i].get('spectral_rolloff_05_hz', np.nan),
                metrics[i].get('low_band_energy_ratio', np.nan),
                metrics[i].get('low_band_periodicity', np.nan),
                metrics[i].get('dynamic_range_db', np.nan),
                metrics[i].get('crest_factor_db', np.nan),
                metrics[i].get('onset_density', np.nan),
                metrics[i].get('onset_magnitude', np.nan),
                metrics[i].get('rhythmic_entropy', np.nan),
                metrics[i].get('band_flux_low', np.nan),
                metrics[i].get('band_flux_mid', np.nan),
                metrics[i].get('band_flux_high', np.nan),
            ]
            for i in idx
        ], dtype=object)

        hover = (
            'genre=%{customdata[0]}<br>'
            'mix=%{customdata[1]}<br>'
            'track=%{customdata[2]}<br>'
            'title=%{customdata[3]}<br>'
            'artists=%{customdata[4]}<br>'
            'energy=%{customdata[5]:.1f}<br>'
            'bpm=%{customdata[6]:.2f}<br>'
            'centroid=%{customdata[7]:.0f} Hz<br>'
            'tilt=%{customdata[8]:.2f} dB/oct<br>'
            'rolloff_05=%{customdata[9]:.0f} Hz<br>'
            'low_ratio=%{customdata[10]:.3f}<br>'
            'low_periodicity=%{customdata[11]:.3f}<br>'
            'dyn_range=%{customdata[12]:.2f} dB<br>'
            'crest=%{customdata[13]:.2f} dB<br>'
            'onset_density=%{customdata[14]:.3f}/s<br>'
            'onset_magnitude=%{customdata[15]:.3f}<br>'
            'rhythmic_entropy=%{customdata[16]:.3f}<br>'
            'band_flux_low=%{customdata[17]:.3f}<br>'
            'band_flux_mid=%{customdata[18]:.3f}<br>'
            'band_flux_high=%{customdata[19]:.3f}<extra></extra>'
        )

        fig.add_trace(
            go.Scatter(
                x=xj,
                y=values[idx],
                mode='markers',
                name=genre,
                legendgroup=f'genre::{genre}',
                showlegend=(f_idx == 0),
                marker=dict(color=genre_colors[genre], size=7, line=dict(color='black', width=0.4), opacity=0.82),
                customdata=custom,
                hovertemplate=hover,
            ),
            row=row_i,
            col=col_i,
        )

    mean_x = []
    mean_y = []
    for e in unique_energy:
        mask = (energy == e) & np.isfinite(values)
        if np.any(mask):
            mean_x.append(float(e))
            mean_y.append(float(np.mean(values[mask])))

    if mean_x:
        fig.add_trace(
            go.Scatter(
                x=np.asarray(mean_x, dtype=np.float64),
                y=np.asarray(mean_y, dtype=np.float64),
                mode='lines+markers',
                name='combined mean',
                legendgroup='combined_mean',
                showlegend=(f_idx == 0),
                line=dict(color='black', width=2),
                marker=dict(size=6),
                hovertemplate=f'energy=%{{x:.1f}}<br>{feature_key}=%{{y:{value_fmt}}}<extra></extra>',
            ),
            row=row_i,
            col=col_i,
        )

    fig.update_xaxes(
        title_text='Labeled Energy Tag',
        tickmode='array',
        tickvals=unique_energy.tolist(),
        row=row_i,
        col=col_i,
    )
    fig.update_yaxes(title_text=y_label, row=row_i, col=col_i)

fig.update_layout(
    title='Aries + Ara Mixes: Audio Features vs Labeled Energy (Genre Colors)',
    width=1320,
    height=max(420, int(250 * rows)),
    template='plotly_white',
    legend=dict(title='Genre'),
)

fig.show()

by_mix = {mix: int(np.sum(mix_names == mix)) for mix in sorted(set(mix_names.tolist()))}
print(f'Plotted {len(metrics)} tracks from mixes: {by_mix}')
print(
    f'Runtime settings: sr={ANALYSIS_SAMPLE_RATE}, n_fft={ANALYSIS_N_FFT}, hop={ANALYSIS_HOP}, '
    f'duration={ANALYSIS_SECONDS if ANALYSIS_SECONDS is not None else "full"}s, cache_hits={cache_hits}'
)
for mix_name, metadata_path in metadata_paths.items():
    print(f'  - {mix_name}: {metadata_path}')

if failed:
    print(f'Skipped {len(failed)} track(s):')
    for mix_name, name, reason in failed[:10]:
        print(f'  - [{mix_name}] {name}: {reason}')
    if len(failed) > 10:
        print(f'  ... and {len(failed) - 10} more')

print('Tip: hover any point to inspect track details; click to pin the tooltip.')


Plotted 48 tracks from mixes: {'ara': 23, 'aries': 25}
Runtime settings: sr=22050, n_fft=4096, hop=512, duration=fulls, cache_hits=0
  - aries: /Users/josephdaher/Git Repositories/djprojectexploration/music/aries-mix/aries_mix_tracks.csv
  - ara: /Users/josephdaher/Git Repositories/djprojectexploration/music/ara-mix/ara_mix_tracks.csv
Tip: hover any point to inspect track details; click to pin the tooltip.


In [5]:
# Advanced groove/structure metrics vs labeled energy (Aries + Ara, genre-colored)
from pathlib import Path
import csv
import math

import numpy as np
import librosa
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from plotly.colors import qualitative as plotly_qual

MIX_METADATA_CANDIDATES = {
    'aries': [
        Path('music/aries-mix/aries_mix_tracks.csv'),
        Path('../music/aries-mix/aries_mix_tracks.csv'),
    ],
    'ara': [
        Path('music/ara-mix/ara_mix_tracks.csv'),
        Path('../music/ara-mix/ara_mix_tracks.csv'),
    ],
}

# Speed / quality knobs
FAST_ANALYSIS = False
ANALYSIS_SECONDS = 90.0 if FAST_ANALYSIS else None
ANALYSIS_SAMPLE_RATE = 22050
ANALYSIS_N_FFT = 2048
ANALYSIS_HOP = 1024
EPS = 1e-12

# Metric config
SUBDIVISIONS_PER_BEAT = 4
BASSLINE_BAND_HZ = (60.0, 250.0)
SECTION_SMOOTH_SEC = 8.0
SECTION_PEAK_SEC = 12.0
SECTION_WAIT_SEC = 8.0
RHYTHM_ENTROPY_PHASE_BINS = 16

METRIC_SPECS = [
    ('downbeat_strength_ratio', 'Downbeat Strength Ratio vs Energy', 'Downbeat Strength Ratio', '.3f'),
    ('phrase_consistency_4bar', 'Phrase Consistency (4-Bar) vs Energy', 'Phrase Consistency 4-Bar', '.3f'),
    ('phrase_consistency_1bar', 'Phrase Consistency (1-Bar) vs Energy', 'Phrase Consistency 1-Bar', '.3f'),
    ('hpss_percussive_ratio', 'HPSS Percussive Ratio vs Energy', 'HPSS Percussive Ratio', '.3f'),
    ('bassline_onset_density', 'Bassline Onset Density vs Energy', 'Bassline Onset Density (onsets / sec)', '.3f'),
    ('spectral_spread_hz', 'Spectral Spread vs Energy', 'Spectral Spread (Hz)', '.1f'),
    ('section_transition_rate', 'Section Transition Rate vs Energy', 'Section Transition Rate (events / min)', '.3f'),
]


def _resolve_first_existing(candidates: list[Path]) -> Path:
    if 'first_existing_path' in globals():
        return first_existing_path(candidates)
    for c in candidates:
        r = c.expanduser().resolve()
        if r.exists():
            return r
    tried = [str(c.expanduser().resolve()) for c in candidates]
    raise FileNotFoundError(f'Could not find any candidate path. Tried: {tried}')


def _safe_float(x):
    try:
        if x is None:
            return None
        s = str(x).strip()
        if s == '':
            return None
        return float(s)
    except Exception:
        return None


def _resolve_track_audio_path(metadata_path: Path, mp3_name: str) -> Path:
    base_dir = metadata_path.parent.resolve()
    candidates = [
        base_dir / mp3_name,
        Path('music/aries-mix') / mp3_name,
        Path('../music/aries-mix') / mp3_name,
        Path('music/ara-mix') / mp3_name,
        Path('../music/ara-mix') / mp3_name,
    ]
    for c in candidates:
        r = c.expanduser().resolve()
        if r.exists():
            return r
    tried = [str(c.expanduser().resolve()) for c in candidates]
    raise FileNotFoundError(f'Could not resolve audio for {mp3_name}. Tried: {tried}')


def _build_beat_grid(duration_sec: float, bpm: float, onset_anchor_sec: float | None = None) -> np.ndarray:
    if bpm <= 0 or not np.isfinite(bpm):
        return np.array([], dtype=np.float64)

    period = 60.0 / float(bpm)
    first = 0.0 if onset_anchor_sec is None else float(max(0.0, onset_anchor_sec))
    first = min(first, max(0.0, float(duration_sec)))

    while first - period >= 0.0:
        first -= period

    beats = np.arange(first, float(duration_sec) + period, period, dtype=np.float64)
    beats = beats[(beats >= 0.0) & (beats <= float(duration_sec))]
    return beats


def _pool_subdivisions(
    onset_times: np.ndarray,
    onset_env: np.ndarray,
    beat_times: np.ndarray,
    subdivisions: int,
) -> np.ndarray:
    times = np.asarray(onset_times, dtype=np.float64).reshape(-1)
    env = np.asarray(onset_env, dtype=np.float64).reshape(-1)
    bt = np.asarray(beat_times, dtype=np.float64).reshape(-1)

    if bt.size < 2 or times.size == 0 or env.size == 0:
        return np.zeros((0, int(subdivisions)), dtype=np.float64)

    rows = []
    for i in range(bt.size - 1):
        start = float(bt[i])
        end = float(bt[i + 1])
        if end <= start:
            continue

        edges = np.linspace(start, end, int(subdivisions) + 1, dtype=np.float64)
        vals = np.zeros(int(subdivisions), dtype=np.float64)

        for s in range(int(subdivisions)):
            l = float(edges[s])
            r = float(edges[s + 1])
            mask = (times >= l) & (times < r)
            if np.any(mask):
                vals[s] = float(np.mean(env[mask]))
            else:
                mid = 0.5 * (l + r)
                vals[s] = float(np.interp(mid, times, env))
        rows.append(vals)

    if not rows:
        return np.zeros((0, int(subdivisions)), dtype=np.float64)
    return np.vstack(rows)


def _mean_cosine_to_template(vectors: np.ndarray, eps: float) -> float:
    V = np.asarray(vectors, dtype=np.float64)
    if V.ndim != 2 or V.shape[0] < 2:
        return float('nan')

    template = np.median(V, axis=0)
    norm_t = float(np.linalg.norm(template))
    norms = np.linalg.norm(V, axis=1)

    valid = (norms > float(eps)) & np.isfinite(norms)
    if not np.any(valid) or norm_t <= float(eps) or not np.isfinite(norm_t):
        return float('nan')

    sims = np.sum(V[valid] * template[None, :], axis=1) / (norms[valid] * norm_t + float(eps))
    if sims.size == 0:
        return float('nan')
    return float(np.mean(sims))


def _estimate_section_transition_rate(
    audio: np.ndarray,
    sr: int,
    *,
    n_fft: int,
    hop_length: int,
    smooth_sec: float,
    peak_sec: float,
    wait_sec: float,
    eps: float,
) -> float:
    mfcc = librosa.feature.mfcc(
        y=audio.astype(np.float32),
        sr=int(sr),
        n_mfcc=20,
        n_fft=int(n_fft),
        hop_length=int(hop_length),
    ).astype(np.float64)

    if mfcc.shape[1] < 4:
        return 0.0

    novelty = np.linalg.norm(np.diff(mfcc, axis=1), axis=0)
    if novelty.size < 3:
        return 0.0

    fps = float(sr) / float(hop_length)
    smooth_frames = max(3, int(round(float(smooth_sec) * fps)))
    kernel = np.ones(smooth_frames, dtype=np.float64)
    kernel /= float(np.sum(kernel))
    novelty_s = np.convolve(novelty, kernel, mode='same')

    if not np.any(np.isfinite(novelty_s)) or float(np.std(novelty_s)) <= 0.0:
        return 0.0

    peak_frames = max(1, int(round(float(peak_sec) * fps)))
    wait_frames = max(1, int(round(float(wait_sec) * fps)))
    delta = 0.5 * float(np.std(novelty_s))

    peaks = librosa.util.peak_pick(
        novelty_s,
        pre_max=peak_frames,
        post_max=peak_frames,
        pre_avg=peak_frames,
        post_avg=peak_frames,
        delta=delta,
        wait=wait_frames,
    )

    duration_min = (float(audio.size) / float(sr)) / 60.0
    return float(len(peaks)) / max(duration_min, float(eps))


def _compute_advanced_metrics(
    audio: np.ndarray,
    sr: int,
    *,
    n_fft: int,
    hop_length: int,
    bpm_hint: float | None,
    onset_anchor_sec: float | None,
    subdivisions_per_beat: int,
    bassline_band_hz: tuple[float, float],
    rhythm_entropy_phase_bins: int,
    section_smooth_sec: float,
    section_peak_sec: float,
    section_wait_sec: float,
    eps: float,
) -> dict:
    stft_mag = np.abs(
        librosa.stft(audio.astype(np.float32), n_fft=int(n_fft), hop_length=int(hop_length), center=True)
    ).astype(np.float64)
    power = np.square(stft_mag) + float(eps)
    freqs = librosa.fft_frequencies(sr=sr, n_fft=int(n_fft)).astype(np.float64)

    onset_env = librosa.onset.onset_strength(
        y=audio.astype(np.float32),
        sr=int(sr),
        hop_length=int(hop_length),
    ).astype(np.float64)
    onset_env = np.maximum(onset_env, 0.0)
    onset_times = librosa.times_like(onset_env, sr=int(sr), hop_length=int(hop_length)).astype(np.float64)

    # Beat grid from metadata BPM if present, else fallback to beat tracker.
    beat_times = np.array([], dtype=np.float64)
    bpm_used = None
    if bpm_hint is not None and np.isfinite(bpm_hint) and bpm_hint > 0:
        bpm_used = float(bpm_hint)
        beat_times = _build_beat_grid(
            duration_sec=float(audio.size) / float(sr),
            bpm=float(bpm_used),
            onset_anchor_sec=onset_anchor_sec,
        )
    else:
        tempo_est, beat_frames = librosa.beat.beat_track(
            onset_envelope=onset_env.astype(np.float32),
            sr=int(sr),
            hop_length=int(hop_length),
            trim=False,
        )
        beat_frames = np.asarray(beat_frames, dtype=np.int64).reshape(-1)
        if beat_frames.size >= 2:
            beat_times = librosa.frames_to_time(beat_frames, sr=int(sr), hop_length=int(hop_length)).astype(np.float64)
            bpm_used = float(tempo_est) if np.isfinite(tempo_est) and tempo_est > 0 else None
        elif np.isfinite(tempo_est) and tempo_est > 0:
            bpm_used = float(tempo_est)
            beat_times = _build_beat_grid(
                duration_sec=float(audio.size) / float(sr),
                bpm=float(tempo_est),
                onset_anchor_sec=0.0,
            )

    # Downbeat strength ratio.
    if beat_times.size >= 4 and onset_times.size > 0:
        beat_strengths = np.interp(beat_times, onset_times, onset_env)
        idx = np.arange(beat_strengths.size, dtype=np.int64)
        down = beat_strengths[(idx % 4) == 0]
        other = beat_strengths[(idx % 4) != 0]
        downbeat_strength_ratio = float(np.mean(down) / (np.mean(other) + float(eps))) if other.size else float('nan')
    else:
        downbeat_strength_ratio = float('nan')

    # Phrase consistency from beat-synchronous subdivision profiles.
    phrase_consistency_1bar = float('nan')
    phrase_consistency_4bar = float('nan')
    if beat_times.size >= 6:
        beat_sub = _pool_subdivisions(onset_times, onset_env, beat_times, subdivisions=int(subdivisions_per_beat))
        n_beats = beat_sub.shape[0]
        n_bars = int(n_beats // 4)
        if n_bars >= 2:
            bars = beat_sub[: n_bars * 4, :].reshape(n_bars, 4 * int(subdivisions_per_beat))
            phrase_consistency_1bar = _mean_cosine_to_template(bars, eps=float(eps))

            n_phrases = int(n_bars // 4)
            if n_phrases >= 2:
                phrases = bars[: n_phrases * 4, :].reshape(n_phrases, 4 * 4 * int(subdivisions_per_beat))
                phrase_consistency_4bar = _mean_cosine_to_template(phrases, eps=float(eps))

    # HPSS percussive ratio.
    harmonic, percussive = librosa.effects.hpss(audio.astype(np.float32))
    pow_total = float(np.mean(np.square(audio.astype(np.float64))))
    pow_perc = float(np.mean(np.square(np.asarray(percussive, dtype=np.float64))))
    hpss_percussive_ratio = float(pow_perc / (pow_total + float(eps)))

    # Bassline onset density from band-limited spectral flux proxy.
    bass_mask = (freqs >= float(bassline_band_hz[0])) & (freqs <= float(bassline_band_hz[1]))
    if np.any(bass_mask):
        bass_energy = np.mean(power[bass_mask, :], axis=0)
        bass_flux = np.maximum(np.diff(bass_energy), 0.0)
        bass_onsets = librosa.onset.onset_detect(
            onset_envelope=bass_flux.astype(np.float32),
            sr=int(sr),
            hop_length=int(hop_length),
            units='frames',
            backtrack=False,
        )
        bassline_onset_density = float(len(bass_onsets)) / max(float(audio.size) / float(sr), float(eps))
    else:
        bassline_onset_density = float('nan')

    # Spectral spread.
    centroid = np.sum(freqs[:, None] * stft_mag, axis=0) / (np.sum(stft_mag, axis=0) + float(eps))
    spread = np.sqrt(
        np.sum(np.square(freqs[:, None] - centroid[None, :]) * stft_mag, axis=0)
        / (np.sum(stft_mag, axis=0) + float(eps))
    )
    spectral_spread_hz = float(np.nanmedian(spread)) if spread.size else float('nan')

    # Section transition rate (major novelty peaks per minute).
    section_transition_rate = _estimate_section_transition_rate(
        audio,
        sr,
        n_fft=int(n_fft),
        hop_length=int(hop_length),
        smooth_sec=float(section_smooth_sec),
        peak_sec=float(section_peak_sec),
        wait_sec=float(section_wait_sec),
        eps=float(eps),
    )

    # Optional rhythmic entropy from beat phase histogram (useful diagnostic).
    rhythmic_entropy = float('nan')
    if onset_env.size > 0:
        if bpm_used is not None and np.isfinite(bpm_used) and bpm_used > 0:
            beat_period = 60.0 / float(bpm_used)
            phase = np.mod(onset_times, beat_period) / max(beat_period, float(eps))
            bins = np.linspace(0.0, 1.0, int(rhythm_entropy_phase_bins) + 1)
            hist, _ = np.histogram(phase, bins=bins, weights=onset_env)
            p = hist.astype(np.float64)
            s = float(np.sum(p))
            if s > float(eps):
                p = p / s
                p = p[p > 0]
                if p.size:
                    h = -float(np.sum(p * np.log(p + float(eps))))
                    hmax = math.log(float(max(2, int(rhythm_entropy_phase_bins))))
                    rhythmic_entropy = float(np.clip(h / max(hmax, float(eps)), 0.0, 1.0))

    return {
        'downbeat_strength_ratio': float(downbeat_strength_ratio),
        'phrase_consistency_4bar': float(phrase_consistency_4bar),
        'phrase_consistency_1bar': float(phrase_consistency_1bar),
        'hpss_percussive_ratio': float(hpss_percussive_ratio),
        'bassline_onset_density': float(bassline_onset_density),
        'spectral_spread_hz': float(spectral_spread_hz),
        'section_transition_rate': float(section_transition_rate),
        'rhythmic_entropy_debug': float(rhythmic_entropy),
    }


if '_aries_ara_advanced_metric_cache_v1' not in globals() or not isinstance(_aries_ara_advanced_metric_cache_v1, dict):
    _aries_ara_advanced_metric_cache_v1 = {}

all_rows = []
metadata_paths = {}
for mix_name, candidates in MIX_METADATA_CANDIDATES.items():
    metadata_path = _resolve_first_existing(candidates)
    metadata_paths[mix_name] = metadata_path
    with metadata_path.open('r', newline='') as f:
        reader = csv.DictReader(f)
        for row in reader:
            row2 = dict(row)
            row2['mix_name'] = mix_name
            row2['metadata_path'] = str(metadata_path)
            all_rows.append(row2)

metrics = []
failed = []
cache_hits = 0

for row in all_rows:
    mix_name = str(row.get('mix_name', '')).strip() or 'unknown'
    metadata_path = Path(str(row.get('metadata_path', '')).strip())

    mp3_name = str(row.get('mp3_name', '')).strip()
    title = str(row.get('title', '')).strip() or mp3_name
    artists = str(row.get('artists', '')).strip()
    genre = (str(row.get('genre', '')).strip() or 'unknown').lower()
    track_number = str(row.get('track_number', '')).strip()

    energy_val = _safe_float(row.get('energy'))
    bpm_hint = _safe_float(row.get('bpm'))
    onset_anchor = _safe_float(row.get('onset-time'))

    if not mp3_name or energy_val is None:
        failed.append((mix_name, title or mp3_name or '<unknown>', 'missing mp3_name or energy'))
        continue

    try:
        track_path = _resolve_track_audio_path(metadata_path, mp3_name)

        cache_key = (
            mix_name,
            str(track_path),
            float(energy_val),
            None if bpm_hint is None else float(bpm_hint),
            None if onset_anchor is None else float(onset_anchor),
            int(ANALYSIS_SAMPLE_RATE),
            int(ANALYSIS_N_FFT),
            int(ANALYSIS_HOP),
            None if ANALYSIS_SECONDS is None else float(ANALYSIS_SECONDS),
            int(SUBDIVISIONS_PER_BEAT),
            tuple(float(v) for v in BASSLINE_BAND_HZ),
            float(SECTION_SMOOTH_SEC),
            float(SECTION_PEAK_SEC),
            float(SECTION_WAIT_SEC),
            int(RHYTHM_ENTROPY_PHASE_BINS),
        )

        if cache_key in _aries_ara_advanced_metric_cache_v1:
            entry = dict(_aries_ara_advanced_metric_cache_v1[cache_key])
            entry['title'] = title
            entry['artists'] = artists
            entry['genre'] = genre
            entry['track_number'] = track_number
            entry['mp3_name'] = mp3_name
            metrics.append(entry)
            cache_hits += 1
            continue

        audio, sr = librosa.load(
            str(track_path),
            sr=int(ANALYSIS_SAMPLE_RATE),
            mono=True,
            duration=ANALYSIS_SECONDS,
        )
        audio = np.asarray(audio, dtype=np.float32)
        if audio.size < int(ANALYSIS_N_FFT):
            failed.append((mix_name, title, f'audio too short ({audio.size} samples)'))
            continue

        vals = _compute_advanced_metrics(
            audio,
            int(sr),
            n_fft=int(ANALYSIS_N_FFT),
            hop_length=int(ANALYSIS_HOP),
            bpm_hint=bpm_hint,
            onset_anchor_sec=onset_anchor,
            subdivisions_per_beat=int(SUBDIVISIONS_PER_BEAT),
            bassline_band_hz=BASSLINE_BAND_HZ,
            rhythm_entropy_phase_bins=int(RHYTHM_ENTROPY_PHASE_BINS),
            section_smooth_sec=float(SECTION_SMOOTH_SEC),
            section_peak_sec=float(SECTION_PEAK_SEC),
            section_wait_sec=float(SECTION_WAIT_SEC),
            eps=float(EPS),
        )

        entry = {
            'mix_name': mix_name,
            'track_number': track_number,
            'title': title,
            'artists': artists,
            'genre': genre,
            'mp3_name': mp3_name,
            'energy': float(energy_val),
            'bpm': float(bpm_hint) if bpm_hint is not None else float('nan'),
            **vals,
        }

        # Keep rows even with partial NaNs, but drop fully invalid rows across requested metrics.
        requested_keys = [k for k, _, _, _ in METRIC_SPECS]
        arr = np.asarray([entry.get(k, np.nan) for k in requested_keys], dtype=np.float64)
        if not np.any(np.isfinite(arr)):
            failed.append((mix_name, title, 'all requested metrics non-finite'))
            continue

        metrics.append(entry)
        _aries_ara_advanced_metric_cache_v1[cache_key] = dict(entry)

    except Exception as exc:
        failed.append((mix_name, title, str(exc)))

if len(metrics) == 0:
    raise RuntimeError('No valid tracks available for advanced metric plotting.')

# Stable snapshot for downstream modeling cells.
advanced_feature_metrics = [dict(m) for m in metrics]
advanced_feature_keys = [spec[0] for spec in METRIC_SPECS]

energy = np.array([m['energy'] for m in metrics], dtype=np.float64)
genres = np.array([m.get('genre', 'unknown') or 'unknown' for m in metrics], dtype=object)
mix_names = np.array([m.get('mix_name', 'unknown') or 'unknown' for m in metrics], dtype=object)

unique_energy = np.array(sorted(set(energy.tolist())), dtype=np.float64)
unique_genres = sorted(set(genres.tolist()))

palette = (
    plotly_qual.Plotly
    + plotly_qual.Safe
    + plotly_qual.Dark24
    + plotly_qual.Alphabet
)
genre_colors = {g: palette[i % len(palette)] for i, g in enumerate(unique_genres)}

cols = 2
rows = int(math.ceil(len(METRIC_SPECS) / float(cols)))
fig = make_subplots(
    rows=rows,
    cols=cols,
    subplot_titles=[spec[1] for spec in METRIC_SPECS],
    horizontal_spacing=0.10,
    vertical_spacing=0.10,
)

rng = np.random.default_rng(42)

for m_idx, (metric_key, _title, y_label, value_fmt) in enumerate(METRIC_SPECS):
    row_i = (m_idx // cols) + 1
    col_i = (m_idx % cols) + 1
    values = np.array([m.get(metric_key, np.nan) for m in metrics], dtype=np.float64)

    for genre in unique_genres:
        idx = np.where(genres == genre)[0]
        idx = idx[np.isfinite(values[idx])]
        if idx.size == 0:
            continue

        xj = energy[idx] + rng.uniform(-0.08, 0.08, size=idx.size)

        custom = np.array([
            [
                metrics[i].get('genre', ''),
                metrics[i].get('mix_name', ''),
                metrics[i].get('track_number', ''),
                metrics[i].get('title', ''),
                metrics[i].get('artists', ''),
                metrics[i].get('energy', np.nan),
                metrics[i].get('bpm', np.nan),
                metrics[i].get('downbeat_strength_ratio', np.nan),
                metrics[i].get('phrase_consistency_4bar', np.nan),
                metrics[i].get('phrase_consistency_1bar', np.nan),
                metrics[i].get('hpss_percussive_ratio', np.nan),
                metrics[i].get('bassline_onset_density', np.nan),
                metrics[i].get('spectral_spread_hz', np.nan),
                metrics[i].get('section_transition_rate', np.nan),
            ]
            for i in idx
        ], dtype=object)

        hover = (
            'genre=%{customdata[0]}<br>'
            'mix=%{customdata[1]}<br>'
            'track=%{customdata[2]}<br>'
            'title=%{customdata[3]}<br>'
            'artists=%{customdata[4]}<br>'
            'energy=%{customdata[5]:.1f}<br>'
            'bpm=%{customdata[6]:.2f}<br>'
            'downbeat_ratio=%{customdata[7]:.3f}<br>'
            'phrase_consistency_4bar=%{customdata[8]:.3f}<br>'
            'phrase_consistency_1bar=%{customdata[9]:.3f}<br>'
            'hpss_percussive_ratio=%{customdata[10]:.3f}<br>'
            'bassline_onset_density=%{customdata[11]:.3f}/s<br>'
            'spectral_spread=%{customdata[12]:.1f} Hz<br>'
            'section_transition_rate=%{customdata[13]:.3f}/min<extra></extra>'
        )

        fig.add_trace(
            go.Scatter(
                x=xj,
                y=values[idx],
                mode='markers',
                name=genre,
                legendgroup=f'genre::{genre}',
                showlegend=(m_idx == 0),
                marker=dict(color=genre_colors[genre], size=7, line=dict(color='black', width=0.4), opacity=0.82),
                customdata=custom,
                hovertemplate=hover,
            ),
            row=row_i,
            col=col_i,
        )

    mean_x = []
    mean_y = []
    for e in unique_energy:
        mask = (energy == e) & np.isfinite(values)
        if np.any(mask):
            mean_x.append(float(e))
            mean_y.append(float(np.mean(values[mask])))

    if mean_x:
        fig.add_trace(
            go.Scatter(
                x=np.asarray(mean_x, dtype=np.float64),
                y=np.asarray(mean_y, dtype=np.float64),
                mode='lines+markers',
                name='combined mean',
                legendgroup='combined_mean',
                showlegend=(m_idx == 0),
                line=dict(color='black', width=2),
                marker=dict(size=6),
                hovertemplate=f'energy=%{{x:.1f}}<br>{metric_key}=%{{y:{value_fmt}}}<extra></extra>',
            ),
            row=row_i,
            col=col_i,
        )

    fig.update_xaxes(
        title_text='Labeled Energy Tag',
        tickmode='array',
        tickvals=unique_energy.tolist(),
        row=row_i,
        col=col_i,
    )
    fig.update_yaxes(title_text=y_label, row=row_i, col=col_i)

fig.update_layout(
    title='Aries + Ara: Advanced Groove/Structure Metrics vs Energy (Genre Colors)',
    width=1320,
    height=max(420, int(260 * rows)),
    template='plotly_white',
    legend=dict(title='Genre'),
)

fig.show()

by_mix = {mix: int(np.sum(mix_names == mix)) for mix in sorted(set(mix_names.tolist()))}
print(f'Plotted {len(metrics)} tracks from mixes: {by_mix}')
print(
    f'Runtime settings: sr={ANALYSIS_SAMPLE_RATE}, n_fft={ANALYSIS_N_FFT}, hop={ANALYSIS_HOP}, '
    f'duration={ANALYSIS_SECONDS if ANALYSIS_SECONDS is not None else "full"}s, cache_hits={cache_hits}'
)
for mix_name, metadata_path in metadata_paths.items():
    print(f'  - {mix_name}: {metadata_path}')

if failed:
    print(f'Skipped {len(failed)} track(s):')
    for mix_name, name, reason in failed[:10]:
        print(f'  - [{mix_name}] {name}: {reason}')
    if len(failed) > 10:
        print(f'  ... and {len(failed) - 10} more')

print('Tip: hover any point to inspect track details; click to pin the tooltip.')


Plotted 46 tracks from mixes: {'ara': 21, 'aries': 25}
Runtime settings: sr=22050, n_fft=2048, hop=1024, duration=fulls, cache_hits=0
  - aries: /Users/josephdaher/Git Repositories/djprojectexploration/music/aries-mix/aries_mix_tracks.csv
  - ara: /Users/josephdaher/Git Repositories/djprojectexploration/music/ara-mix/ara_mix_tracks.csv
Skipped 2 track(s):
  - [ara] Poa Alpina: only 0-dimensional arrays can be converted to Python scalars
  - [ara] Water Slide: only 0-dimensional arrays can be converted to Python scalars
Tip: hover any point to inspect track details; click to pin the tooltip.


In [17]:
# LOUDNESS / RMS vs LABELED ENERGY (ARIES + ARA, GENRE-COLORED)
from pathlib import Path
import csv
import math

import numpy as np
import librosa
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from plotly.colors import qualitative as plotly_qual
import pyloudnorm as pyln

MIX_METADATA_CANDIDATES = {
    'aries': [
        Path('music/aries-mix/aries_mix_tracks.csv'),
        Path('../music/aries-mix/aries_mix_tracks.csv'),
    ],
    'ara': [
        Path('music/ara-mix/ara_mix_tracks.csv'),
        Path('../music/ara-mix/ara_mix_tracks.csv'),
    ],
}

# Speed / quality knobs. Full-track loudness is preferred; set FAST_ANALYSIS=True for quick iteration.
FAST_ANALYSIS = False
LOUDNESS_ANALYSIS_SECONDS = 120.0 if FAST_ANALYSIS else None
LOUDNESS_SAMPLE_RATE = 22050
RMS_FRAME_SEC = 0.050
RMS_HOP_SEC = 0.025
SHORT_TERM_LUFS_WINDOW_SEC = 3.0
SHORT_TERM_LUFS_HOP_SEC = 1.0
EPS = 1e-12

LOUDNESS_RMS_FEATURE_SPECS = [
    ('integrated_loudness_lufs', 'Integrated Loudness vs Energy', 'Integrated Loudness (LUFS)', '.2f'),
    ('short_term_loudness_mean_lufs', 'Short-Term Loudness Mean vs Energy', 'Short-Term Loudness Mean (LUFS)', '.2f'),
    ('short_term_loudness_std_lu', 'Short-Term Loudness Std vs Energy', 'Short-Term Loudness Std (LU)', '.2f'),
    ('short_term_loudness_range_lu', 'Short-Term Loudness Range vs Energy', 'Short-Term Loudness p95-p10 (LU)', '.2f'),
    ('rms_mean_db', 'RMS Mean vs Energy', 'RMS Mean (dBFS)', '.2f'),
    ('rms_median_db', 'RMS Median vs Energy', 'RMS Median (dBFS)', '.2f'),
    ('rms_std_db', 'RMS Std vs Energy', 'RMS Std across frames (dB)', '.2f'),
    ('rms_iqr_db', 'RMS IQR vs Energy', 'RMS Interquartile Range (dB)', '.2f'),
    ('rms_range_db', 'RMS Range vs Energy', 'RMS p95-p10 Range (dB)', '.2f'),
    ('rms_var', 'RMS Variance vs Energy', 'RMS Variance (linear)', '.6f'),
]


def _resolve_first_existing(candidates: list[Path]) -> Path:
    if 'first_existing_path' in globals():
        return first_existing_path(candidates)
    for c in candidates:
        r = c.expanduser().resolve()
        if r.exists():
            return r
    tried = [str(c.expanduser().resolve()) for c in candidates]
    raise FileNotFoundError(f'Could not find any candidate path. Tried: {tried}')


def _safe_float(x):
    try:
        if x is None:
            return None
        s = str(x).strip()
        if s == '':
            return None
        v = float(s)
        return v if np.isfinite(v) else None
    except Exception:
        return None


def _resolve_track_audio_path(metadata_path: Path, mp3_name: str) -> Path:
    if 'resolve_track_audio_path' in globals():
        return resolve_track_audio_path(metadata_path, mp3_name)
    base_dir = metadata_path.parent.resolve()
    candidates = [
        base_dir / mp3_name,
        Path('music/aries-mix') / mp3_name,
        Path('../music/aries-mix') / mp3_name,
        Path('music/ara-mix') / mp3_name,
        Path('../music/ara-mix') / mp3_name,
    ]
    for c in candidates:
        r = c.expanduser().resolve()
        if r.exists():
            return r
    tried = [str(c.expanduser().resolve()) for c in candidates]
    raise FileNotFoundError(f'Could not resolve audio for {mp3_name}. Tried: {tried}')


def _finite_percentile(values: np.ndarray, q: float, fallback: float = float('nan')) -> float:
    arr = np.asarray(values, dtype=np.float64).reshape(-1)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return float(fallback)
    return float(np.percentile(arr, q))


def _safe_integrated_loudness(meter: pyln.Meter, audio: np.ndarray) -> float:
    try:
        value = float(meter.integrated_loudness(np.asarray(audio, dtype=np.float64)))
        return value if np.isfinite(value) else float('nan')
    except Exception:
        return float('nan')


def _frame_rms_features(audio: np.ndarray, sr: int, *, frame_sec: float, hop_sec: float, eps: float) -> dict[str, float]:
    x = np.asarray(audio, dtype=np.float64).reshape(-1)
    frame_len = max(1, int(round(float(frame_sec) * int(sr))))
    hop_len = max(1, int(round(float(hop_sec) * int(sr))))

    if x.size < frame_len:
        rms = np.array([np.sqrt(np.mean(np.square(x)) + float(eps))], dtype=np.float64)
    else:
        frames = librosa.util.frame(x, frame_length=frame_len, hop_length=hop_len).T
        rms = np.sqrt(np.mean(np.square(frames), axis=1) + float(eps))

    rms_db = 20.0 * np.log10(np.maximum(rms, float(eps)))
    return {
        'rms_mean': float(np.mean(rms)),
        'rms_median': float(np.median(rms)),
        'rms_var': float(np.var(rms)),
        'rms_mean_db': float(np.mean(rms_db)),
        'rms_median_db': float(np.median(rms_db)),
        'rms_std_db': float(np.std(rms_db)),
        'rms_p10_db': _finite_percentile(rms_db, 10),
        'rms_p90_db': _finite_percentile(rms_db, 90),
        'rms_iqr_db': _finite_percentile(rms_db, 75) - _finite_percentile(rms_db, 25),
        'rms_range_db': _finite_percentile(rms_db, 95) - _finite_percentile(rms_db, 10),
    }


def _short_term_loudness_features(
    audio: np.ndarray,
    sr: int,
    *,
    window_sec: float,
    hop_sec: float,
) -> dict[str, float]:
    x = np.asarray(audio, dtype=np.float64).reshape(-1)
    meter = pyln.Meter(int(sr))
    integrated = _safe_integrated_loudness(meter, x)

    win = max(1, int(round(float(window_sec) * int(sr))))
    hop = max(1, int(round(float(hop_sec) * int(sr))))
    starts = [0]
    if x.size > win:
        starts = list(range(0, x.size - win + 1, hop))
        if starts[-1] != x.size - win:
            starts.append(x.size - win)

    short_lufs = []
    for start in starts:
        segment = x[start:min(x.size, start + win)]
        if segment.size < max(1, int(0.5 * sr)):
            continue
        value = _safe_integrated_loudness(meter, segment)
        if np.isfinite(value):
            short_lufs.append(value)

    short_lufs = np.asarray(short_lufs, dtype=np.float64)
    if short_lufs.size == 0:
        short_lufs = np.array([integrated], dtype=np.float64) if np.isfinite(integrated) else np.array([], dtype=np.float64)

    p10 = _finite_percentile(short_lufs, 10)
    p25 = _finite_percentile(short_lufs, 25)
    p50 = _finite_percentile(short_lufs, 50)
    p75 = _finite_percentile(short_lufs, 75)
    p90 = _finite_percentile(short_lufs, 90)
    p95 = _finite_percentile(short_lufs, 95)

    return {
        'integrated_loudness_lufs': integrated,
        'short_term_loudness_mean_lufs': float(np.mean(short_lufs)) if short_lufs.size else float('nan'),
        'short_term_loudness_median_lufs': p50,
        'short_term_loudness_std_lu': float(np.std(short_lufs)) if short_lufs.size else float('nan'),
        'short_term_loudness_p10_lufs': p10,
        'short_term_loudness_p90_lufs': p90,
        'short_term_loudness_range_lu': p95 - p10,
        'short_term_loudness_iqr_lu': p75 - p25,
    }


def _compute_loudness_rms_features(audio: np.ndarray, sr: int) -> dict[str, float]:
    audio = np.asarray(audio, dtype=np.float32).reshape(-1)
    if audio.size == 0:
        return {key: float('nan') for key, *_ in LOUDNESS_RMS_FEATURE_SPECS}
    return {
        **_frame_rms_features(audio, sr, frame_sec=RMS_FRAME_SEC, hop_sec=RMS_HOP_SEC, eps=EPS),
        **_short_term_loudness_features(
            audio,
            sr,
            window_sec=SHORT_TERM_LUFS_WINDOW_SEC,
            hop_sec=SHORT_TERM_LUFS_HOP_SEC,
        ),
    }


if '_aries_ara_loudness_rms_cache_v1' not in globals() or not isinstance(_aries_ara_loudness_rms_cache_v1, dict):
    _aries_ara_loudness_rms_cache_v1 = {}

all_rows = []
metadata_paths = {}
for mix_name, candidates in MIX_METADATA_CANDIDATES.items():
    metadata_path = _resolve_first_existing(candidates)
    metadata_paths[mix_name] = metadata_path
    with metadata_path.open('r', newline='') as f:
        reader = csv.DictReader(f)
        for row in reader:
            row2 = dict(row)
            row2['mix_name'] = mix_name
            row2['metadata_path'] = str(metadata_path)
            all_rows.append(row2)

metrics = []
failed = []
cache_hits = 0

for row in all_rows:
    mix_name = str(row.get('mix_name', '')).strip() or 'unknown'
    metadata_path = Path(str(row.get('metadata_path', '')).strip())
    mp3_name = str(row.get('mp3_name', '')).strip()
    title = str(row.get('title', '')).strip() or mp3_name
    artists = str(row.get('artists', '')).strip()
    genre = (str(row.get('genre', '')).strip() or 'unknown').lower()
    track_number = str(row.get('track_number', '')).strip()
    energy_val = _safe_float(row.get('energy'))
    bpm_hint = _safe_float(row.get('bpm'))

    if not mp3_name or energy_val is None:
        failed.append((mix_name, title or mp3_name or '<unknown>', 'missing mp3_name or energy'))
        continue

    try:
        track_path = _resolve_track_audio_path(metadata_path, mp3_name)
        cache_key = (
            mix_name,
            str(track_path),
            float(energy_val),
            int(LOUDNESS_SAMPLE_RATE),
            None if LOUDNESS_ANALYSIS_SECONDS is None else float(LOUDNESS_ANALYSIS_SECONDS),
            float(RMS_FRAME_SEC),
            float(RMS_HOP_SEC),
            float(SHORT_TERM_LUFS_WINDOW_SEC),
            float(SHORT_TERM_LUFS_HOP_SEC),
        )

        if cache_key in _aries_ara_loudness_rms_cache_v1:
            entry = dict(_aries_ara_loudness_rms_cache_v1[cache_key])
            entry.update({
                'title': title,
                'artists': artists,
                'genre': genre,
                'track_number': track_number,
                'mp3_name': mp3_name,
            })
            metrics.append(entry)
            cache_hits += 1
            continue

        audio, sr = librosa.load(
            str(track_path),
            sr=LOUDNESS_SAMPLE_RATE,
            mono=True,
            duration=LOUDNESS_ANALYSIS_SECONDS,
        )
        audio = np.asarray(audio, dtype=np.float32)
        if audio.size < max(1, int(0.5 * LOUDNESS_SAMPLE_RATE)):
            failed.append((mix_name, title, f'audio too short ({audio.size} samples)'))
            continue

        feat = _compute_loudness_rms_features(audio, sr)
        entry = {
            'mix_name': mix_name,
            'track_number': track_number,
            'title': title,
            'artists': artists,
            'genre': genre,
            'mp3_name': mp3_name,
            'bpm': float(bpm_hint) if bpm_hint is not None else float('nan'),
            'energy': float(energy_val),
            **feat,
        }
        metrics.append(entry)
        _aries_ara_loudness_rms_cache_v1[cache_key] = dict(entry)

    except Exception as exc:
        failed.append((mix_name, title, str(exc)))

if len(metrics) == 0:
    raise RuntimeError('No valid tracks available for loudness/RMS vs energy plotting.')

# Stable snapshot for downstream modeling cells.
loudness_rms_feature_metrics = [dict(m) for m in metrics]
loudness_rms_feature_keys = [spec[0] for spec in LOUDNESS_RMS_FEATURE_SPECS]

mix_names = np.array([m['mix_name'] for m in metrics], dtype=object)
genres = np.array([m.get('genre', 'unknown') or 'unknown' for m in metrics], dtype=object)
energy = np.array([m['energy'] for m in metrics], dtype=np.float64)
unique_energy = np.array(sorted(set(energy.tolist())), dtype=np.float64)
unique_genres = sorted(set(genres.tolist()))

palette = (
    plotly_qual.Plotly
    + plotly_qual.Safe
    + plotly_qual.Dark24
    + plotly_qual.Alphabet
)
genre_colors = {g: palette[i % len(palette)] for i, g in enumerate(unique_genres)}

cols = 2
rows_n = int(math.ceil(len(LOUDNESS_RMS_FEATURE_SPECS) / cols))
fig = make_subplots(
    rows=rows_n,
    cols=cols,
    subplot_titles=[spec[1] for spec in LOUDNESS_RMS_FEATURE_SPECS],
    horizontal_spacing=0.09,
    vertical_spacing=0.085,
)

rng = np.random.default_rng(48)

for f_idx, (feature_key, _title, y_label, value_fmt) in enumerate(LOUDNESS_RMS_FEATURE_SPECS):
    row_i = (f_idx // cols) + 1
    col_i = (f_idx % cols) + 1
    values = np.array([m.get(feature_key, np.nan) for m in metrics], dtype=np.float64)

    for genre in unique_genres:
        idx = np.where(genres == genre)[0]
        idx = idx[np.isfinite(values[idx])]
        if idx.size == 0:
            continue

        xj = energy[idx] + rng.uniform(-0.08, 0.08, size=idx.size)
        custom = np.array([
            [
                metrics[i].get('genre', ''),
                metrics[i].get('mix_name', ''),
                metrics[i].get('track_number', ''),
                metrics[i].get('title', ''),
                metrics[i].get('artists', ''),
                metrics[i].get('energy', np.nan),
                metrics[i].get('bpm', np.nan),
                metrics[i].get('integrated_loudness_lufs', np.nan),
                metrics[i].get('short_term_loudness_mean_lufs', np.nan),
                metrics[i].get('short_term_loudness_std_lu', np.nan),
                metrics[i].get('short_term_loudness_range_lu', np.nan),
                metrics[i].get('rms_mean_db', np.nan),
                metrics[i].get('rms_std_db', np.nan),
                metrics[i].get('rms_iqr_db', np.nan),
                metrics[i].get('rms_var', np.nan),
            ]
            for i in idx
        ], dtype=object)

        hover = (
            'genre=%{customdata[0]}<br>'
            'mix=%{customdata[1]}<br>'
            'track=%{customdata[2]}<br>'
            'title=%{customdata[3]}<br>'
            'artists=%{customdata[4]}<br>'
            'energy=%{customdata[5]:.1f}<br>'
            'bpm=%{customdata[6]:.2f}<br>'
            'integrated_lufs=%{customdata[7]:.2f}<br>'
            'short_lufs_mean=%{customdata[8]:.2f}<br>'
            'short_lufs_std=%{customdata[9]:.2f} LU<br>'
            'short_lufs_range=%{customdata[10]:.2f} LU<br>'
            'rms_mean=%{customdata[11]:.2f} dBFS<br>'
            'rms_std=%{customdata[12]:.2f} dB<br>'
            'rms_iqr=%{customdata[13]:.2f} dB<br>'
            'rms_var=%{customdata[14]:.6f}<extra></extra>'
        )

        fig.add_trace(
            go.Scatter(
                x=xj,
                y=values[idx],
                mode='markers',
                name=genre,
                legendgroup=f'genre::{genre}',
                showlegend=(f_idx == 0),
                marker=dict(color=genre_colors[genre], size=7, line=dict(color='black', width=0.4), opacity=0.82),
                customdata=custom,
                hovertemplate=hover,
            ),
            row=row_i,
            col=col_i,
        )

    mean_x = []
    mean_y = []
    for e in unique_energy:
        mask = (energy == e) & np.isfinite(values)
        if np.any(mask):
            mean_x.append(float(e))
            mean_y.append(float(np.mean(values[mask])))

    if mean_x:
        fig.add_trace(
            go.Scatter(
                x=np.asarray(mean_x, dtype=np.float64),
                y=np.asarray(mean_y, dtype=np.float64),
                mode='lines+markers',
                name='combined mean',
                legendgroup='combined_mean',
                showlegend=(f_idx == 0),
                line=dict(color='black', width=2),
                marker=dict(size=6),
                hovertemplate=f'energy=%{{x:.1f}}<br>{feature_key}=%{{y:{value_fmt}}}<extra></extra>',
            ),
            row=row_i,
            col=col_i,
        )

    fig.update_xaxes(
        title_text='Labeled Energy Tag',
        tickmode='array',
        tickvals=unique_energy.tolist(),
        row=row_i,
        col=col_i,
    )
    fig.update_yaxes(title_text=y_label, row=row_i, col=col_i)

fig.update_layout(
    title='Aries + Ara: Loudness / RMS Features vs Labeled Energy (Genre Colors)',
    width=1320,
    height=max(520, int(285 * rows_n)),
    template='plotly_white',
    legend=dict(title='Genre'),
)

fig.show()

by_mix = {mix: int(np.sum(mix_names == mix)) for mix in sorted(set(mix_names.tolist()))}
print(f'Plotted {len(metrics)} tracks from mixes: {by_mix}')
print(
    f'Runtime settings: sr={LOUDNESS_SAMPLE_RATE}, duration={LOUDNESS_ANALYSIS_SECONDS if LOUDNESS_ANALYSIS_SECONDS is not None else "full"}s, '
    f'rms_frame={RMS_FRAME_SEC}s, rms_hop={RMS_HOP_SEC}s, short_lufs_window={SHORT_TERM_LUFS_WINDOW_SEC}s, '
    f'short_lufs_hop={SHORT_TERM_LUFS_HOP_SEC}s, cache_hits={cache_hits}'
)
for mix_name, metadata_path in metadata_paths.items():
    print(f'  - {mix_name}: {metadata_path}')

if failed:
    print(f'Skipped {len(failed)} track(s):')
    for mix_name, name, reason in failed[:10]:
        print(f'  - [{mix_name}] {name}: {reason}')
    if len(failed) > 10:
        print(f'  ... and {len(failed) - 10} more')

print('Tip: hover any point to inspect track details; click to pin the tooltip.')


Plotted 48 tracks from mixes: {'ara': 23, 'aries': 25}
Runtime settings: sr=22050, duration=fulls, rms_frame=0.05s, rms_hop=0.025s, short_lufs_window=3.0s, short_lufs_hop=1.0s, cache_hits=0
  - aries: /Users/josephdaher/Git Repositories/djprojectexploration/music/aries-mix/aries_mix_tracks.csv
  - ara: /Users/josephdaher/Git Repositories/djprojectexploration/music/ara-mix/ara_mix_tracks.csv
Tip: hover any point to inspect track details; click to pin the tooltip.


In [11]:
# Regularized GLM: combine simple + advanced audio features to predict labeled energy
import math
import warnings

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.linear_model import TweedieRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import KFold, RepeatedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

SIMPLE_FEATURE_KEYS_FOR_GLM = [
    'spectral_centroid_hz',
    'spectral_tilt_db_per_oct',
    'spectral_rolloff_05_hz',
    'low_band_energy_ratio',
    'low_band_periodicity',
    'dynamic_range_db',
    'crest_factor_db',
    'onset_density',
    'onset_magnitude',
    'rhythmic_entropy',
    'band_flux_low',
    'band_flux_mid',
    'band_flux_high',
]

ADVANCED_FEATURE_KEYS_FOR_GLM = [
    'downbeat_strength_ratio',
    'phrase_consistency_4bar',
    'phrase_consistency_1bar',
    'hpss_percussive_ratio',
    'bassline_onset_density',
    'spectral_spread_hz',
    'section_transition_rate',
]

GLM_ALPHA_GRID = np.logspace(-3, 2, 26)
GLM_RANDOM_STATE = 42
GLM_MIN_VALID_FEATURES = 4


def _track_key(row: dict) -> tuple[str, str]:
    return (
        str(row.get('mix_name', '')).strip().lower(),
        str(row.get('mp3_name', '')).strip().lower(),
    )


def _safe_float_or_nan(value) -> float:
    try:
        v = float(value)
        return v if np.isfinite(v) else float('nan')
    except Exception:
        return float('nan')


if 'simple_feature_metrics' not in globals():
    raise RuntimeError('Run cell `aries-centroid-tilt-vs-energy` first to populate `simple_feature_metrics`.')
if 'advanced_feature_metrics' not in globals():
    raise RuntimeError('Run cell `advanced-groove-metrics-vs-energy-by-genre` first to populate `advanced_feature_metrics`.')

simple_by_key = {_track_key(m): dict(m) for m in simple_feature_metrics}
advanced_by_key = {_track_key(m): dict(m) for m in advanced_feature_metrics}
common_keys = sorted(set(simple_by_key.keys()) & set(advanced_by_key.keys()))

rows = []
for key in common_keys:
    simple = simple_by_key[key]
    advanced = advanced_by_key[key]
    energy = _safe_float_or_nan(simple.get('energy', advanced.get('energy')))
    if not np.isfinite(energy):
        continue

    row = {
        'key': key,
        'mix_name': simple.get('mix_name', advanced.get('mix_name', '')),
        'track_number': simple.get('track_number', advanced.get('track_number', '')),
        'title': simple.get('title', advanced.get('title', '')),
        'artists': simple.get('artists', advanced.get('artists', '')),
        'genre': simple.get('genre', advanced.get('genre', 'unknown')) or 'unknown',
        'energy': float(energy),
    }
    for f in SIMPLE_FEATURE_KEYS_FOR_GLM:
        row[f] = _safe_float_or_nan(simple.get(f))
    for f in ADVANCED_FEATURE_KEYS_FOR_GLM:
        row[f] = _safe_float_or_nan(advanced.get(f))
    rows.append(row)

feature_names = SIMPLE_FEATURE_KEYS_FOR_GLM + ADVANCED_FEATURE_KEYS_FOR_GLM

if len(rows) < 6:
    raise RuntimeError(f'Need at least 6 joined tracks for CV; found {len(rows)}.')

X_raw = np.array([[r[f] for f in feature_names] for r in rows], dtype=np.float64)
y = np.array([r['energy'] for r in rows], dtype=np.float64)

valid_feature_mask = np.sum(np.isfinite(X_raw), axis=0) >= GLM_MIN_VALID_FEATURES
feature_names = [name for name, keep in zip(feature_names, valid_feature_mask) if keep]
X_raw = X_raw[:, valid_feature_mask]

if X_raw.shape[1] == 0:
    raise RuntimeError('No usable features after finite-value filtering.')

# Drop rows whose target is finite but all retained features are missing.
valid_row_mask = np.any(np.isfinite(X_raw), axis=1) & np.isfinite(y)
X_raw = X_raw[valid_row_mask]
y = y[valid_row_mask]
rows = [r for r, keep in zip(rows, valid_row_mask) if keep]

if len(rows) < 6:
    raise RuntimeError(f'Need at least 6 valid tracks after filtering; found {len(rows)}.')

base_model = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
    TweedieRegressor(
        power=0,
        link='identity',
        alpha=1.0,
        fit_intercept=True,
        max_iter=10000,
        tol=1e-7,
    ),
)

n_splits = min(5, len(rows))
cv = RepeatedKFold(n_splits=n_splits, n_repeats=20, random_state=GLM_RANDOM_STATE)

alpha_scores = []
with warnings.catch_warnings():
    warnings.simplefilter('ignore', ConvergenceWarning)
    for alpha in GLM_ALPHA_GRID:
        fold_mae = []
        for train_idx, test_idx in cv.split(X_raw, y):
            model = clone(base_model)
            model.set_params(tweedieregressor__alpha=float(alpha))
            model.fit(X_raw[train_idx], y[train_idx])
            pred = model.predict(X_raw[test_idx])
            fold_mae.append(mean_absolute_error(y[test_idx], pred))
        alpha_scores.append(float(np.mean(fold_mae)))

alpha_scores = np.asarray(alpha_scores, dtype=np.float64)
best_alpha = float(GLM_ALPHA_GRID[int(np.argmin(alpha_scores))])

# Out-of-fold predictions for the selected alpha using a single non-repeated split for readable diagnostics.
diagnostic_cv = KFold(n_splits=n_splits, shuffle=True, random_state=GLM_RANDOM_STATE)
oof_pred = np.full_like(y, np.nan, dtype=np.float64)
with warnings.catch_warnings():
    warnings.simplefilter('ignore', ConvergenceWarning)
    for train_idx, test_idx in diagnostic_cv.split(X_raw, y):
        model = clone(base_model)
        model.set_params(tweedieregressor__alpha=best_alpha)
        model.fit(X_raw[train_idx], y[train_idx])
        oof_pred[test_idx] = model.predict(X_raw[test_idx])

final_model = clone(base_model)
final_model.set_params(tweedieregressor__alpha=best_alpha)
with warnings.catch_warnings():
    warnings.simplefilter('ignore', ConvergenceWarning)
    final_model.fit(X_raw, y)

coef = final_model.named_steps['tweedieregressor'].coef_.astype(np.float64)
intercept = float(final_model.named_steps['tweedieregressor'].intercept_)
rank_idx = np.argsort(np.abs(coef))[::-1]

baseline_pred = np.full_like(y, float(np.mean(y)), dtype=np.float64)
baseline_mae = float(mean_absolute_error(y, baseline_pred))
oof_mae = float(mean_absolute_error(y, oof_pred))
oof_r2 = float(r2_score(y, oof_pred)) if len(np.unique(y)) > 1 else float('nan')
train_pred = final_model.predict(X_raw)
train_mae = float(mean_absolute_error(y, train_pred))
train_r2 = float(r2_score(y, train_pred)) if len(np.unique(y)) > 1 else float('nan')

custom = np.array([
    [
        r.get('genre', ''),
        r.get('mix_name', ''),
        r.get('track_number', ''),
        r.get('title', ''),
        r.get('artists', ''),
        r.get('energy', np.nan),
        oof_pred[i],
        train_pred[i],
    ]
    for i, r in enumerate(rows)
], dtype=object)

# Coefficient diagnostics: standardized coefficients are directly comparable because
# the model uses median imputation followed by StandardScaler before the GLM.
feature_group_by_name = {
    **{name: 'Simple spectral/groove' for name in SIMPLE_FEATURE_KEYS_FOR_GLM},
    **{name: 'Advanced rhythm/structure' for name in ADVANCED_FEATURE_KEYS_FOR_GLM},
}
feature_label_by_name = {
    'spectral_centroid_hz': 'Spectral centroid',
    'spectral_tilt_db_per_oct': 'Spectral tilt',
    'spectral_rolloff_05_hz': 'Spectral rolloff 50%',
    'low_band_energy_ratio': 'Low-band energy ratio',
    'low_band_periodicity': 'Low-band periodicity',
    'dynamic_range_db': 'Dynamic range',
    'crest_factor_db': 'Crest factor',
    'onset_density': 'Onset density',
    'onset_magnitude': 'Onset magnitude',
    'rhythmic_entropy': 'Rhythmic entropy',
    'band_flux_low': 'Low-band flux',
    'band_flux_mid': 'Mid-band flux',
    'band_flux_high': 'High-band flux',
    'downbeat_strength_ratio': 'Downbeat strength ratio',
    'phrase_consistency_4bar': 'Phrase consistency, 4 bar',
    'phrase_consistency_1bar': 'Phrase consistency, 1 bar',
    'hpss_percussive_ratio': 'HPSS percussive ratio',
    'bassline_onset_density': 'Bassline onset density',
    'spectral_spread_hz': 'Spectral spread',
    'section_transition_rate': 'Section transition rate',
}

glm_coefficient_summary = []
for name, value in zip(feature_names, coef):
    group = feature_group_by_name.get(name, 'Other')
    label = feature_label_by_name.get(name, name.replace('_', ' ').title())
    glm_coefficient_summary.append({
        'feature': name,
        'label': label,
        'group': group,
        'coefficient': float(value),
        'abs_coefficient': float(abs(value)),
        'direction': 'Positive' if value > 0 else ('Negative' if value < 0 else 'Zero'),
        'finite_count': int(np.sum(np.isfinite(X_raw[:, feature_names.index(name)]))),
    })

glm_coefficient_summary = sorted(
    glm_coefficient_summary,
    key=lambda row: (row['abs_coefficient'], row['label'].lower()),
    reverse=True,
)

plot_rows = list(reversed(glm_coefficient_summary))
coef_colors = [
    '#0b6b57' if row['coefficient'] >= 0 else '#b7472a'
    for row in plot_rows
]
coef_hover = [
    (
        f"<b>{row['label']}</b><br>"
        f"Feature key: {row['feature']}<br>"
        f"Group: {row['group']}<br>"
        f"Standardized coefficient: {row['coefficient']:.4f}<br>"
        f"Finite tracks before imputation: {row['finite_count']}<extra></extra>"
    )
    for row in plot_rows
]

coef_fig = go.Figure()
coef_fig.add_trace(go.Bar(
    x=[row['coefficient'] for row in plot_rows],
    y=[row['label'] for row in plot_rows],
    orientation='h',
    marker=dict(
        color=coef_colors,
        line=dict(color='rgba(20, 28, 38, 0.35)', width=0.8),
    ),
    customdata=np.array([[row['feature'], row['group'], row['finite_count']] for row in plot_rows], dtype=object),
    hovertemplate=coef_hover,
    showlegend=False,
))
coef_fig.add_vline(x=0, line_width=1.5, line_color='rgba(23, 32, 42, 0.7)')

max_abs_coef = max([row['abs_coefficient'] for row in glm_coefficient_summary] + [0.05])
coef_fig.update_xaxes(
    title='Standardized coefficient: energy points per +1 SD feature change',
    zeroline=False,
    range=[-max_abs_coef * 1.18, max_abs_coef * 1.18],
    gridcolor='rgba(133, 146, 166, 0.22)',
)
coef_fig.update_yaxes(title='', automargin=True)
coef_fig.update_layout(
    title=(
        'Regularized GLM Coefficients for Energy Prediction'
        f'<br><sup>Gaussian GLM / ridge penalty, alpha={best_alpha:.4g}; '
        f'n={len(rows)} tracks; OOF MAE={oof_mae:.3f} vs baseline MAE={baseline_mae:.3f}; '
        f'OOF R2={oof_r2:.3f}</sup>'
    ),
    height=max(520, 30 * len(plot_rows) + 180),
    width=1100,
    margin=dict(l=230, r=40, t=92, b=70),
    template='plotly_white',
    paper_bgcolor='#fbfcfe',
    plot_bgcolor='#fbfcfe',
    font=dict(size=12, color='#17202a'),
    annotations=[
        dict(
            x=0.01,
            y=1.04,
            xref='paper',
            yref='paper',
            text='<span style="color:#0b6b57"><b>Positive</b></span> = higher predicted energy; '
                 '<span style="color:#b7472a"><b>Negative</b></span> = lower predicted energy',
            showarrow=False,
            align='left',
            font=dict(size=12),
        )
    ],
)
coef_fig.show()

# Display a compact, sortable-by-importance coefficient table as a Plotly table.
coef_table_rows = glm_coefficient_summary
rank_values = list(range(1, len(coef_table_rows) + 1))
coef_table_fig = go.Figure(data=[go.Table(
    columnwidth=[0.5, 2.4, 1.35, 1.0, 1.05, 0.75, 1.6],
    header=dict(
        values=[
            '<b>Rank</b>',
            '<b>Feature</b>',
            '<b>Group</b>',
            '<b>Direction</b>',
            '<b>Std. coef</b>',
            '<b>|coef|</b>',
            '<b>Finite tracks</b>',
        ],
        fill_color='#17202a',
        font=dict(color='white', size=12),
        align=['right', 'left', 'left', 'center', 'right', 'right', 'right'],
        height=30,
    ),
    cells=dict(
        values=[
            rank_values,
            [row['label'] + '<br><span style="color:#6b7280">' + row['feature'] + '</span>' for row in coef_table_rows],
            [row['group'] for row in coef_table_rows],
            [row['direction'] for row in coef_table_rows],
            [f"{row['coefficient']:+.4f}" for row in coef_table_rows],
            [f"{row['abs_coefficient']:.4f}" for row in coef_table_rows],
            [row['finite_count'] for row in coef_table_rows],
        ],
        fill_color=[
            ['#f8fafc' if i % 2 == 0 else '#eef2f7' for i in range(len(coef_table_rows))],
            ['#f8fafc' if i % 2 == 0 else '#eef2f7' for i in range(len(coef_table_rows))],
            ['#f8fafc' if i % 2 == 0 else '#eef2f7' for i in range(len(coef_table_rows))],
            ['#e9f6f2' if row['coefficient'] > 0 else ('#fbeee9' if row['coefficient'] < 0 else '#f2f4f7') for row in coef_table_rows],
            ['#f8fafc' if i % 2 == 0 else '#eef2f7' for i in range(len(coef_table_rows))],
            ['#f8fafc' if i % 2 == 0 else '#eef2f7' for i in range(len(coef_table_rows))],
            ['#f8fafc' if i % 2 == 0 else '#eef2f7' for i in range(len(coef_table_rows))],
        ],
        font=dict(color='#17202a', size=11),
        align=['right', 'left', 'left', 'center', 'right', 'right', 'right'],
        height=34,
    ),
)])
coef_table_fig.update_layout(
    title=(
        'GLM Coefficient Table, Ranked by Absolute Standardized Effect'
        '<br><sup>Coefficients are descriptive regularized effects, not causal estimates. '
        'Correlated features can share or swap weight.</sup>'
    ),
    height=max(520, 34 * len(coef_table_rows) + 130),
    width=1100,
    margin=dict(l=20, r=20, t=82, b=20),
    paper_bgcolor='#fbfcfe',
)
coef_table_fig.show()

print(f'Joined tracks used by GLM: {len(rows)}')
print(f'Usable features retained: {len(feature_names)} / {len(SIMPLE_FEATURE_KEYS_FOR_GLM) + len(ADVANCED_FEATURE_KEYS_FOR_GLM)}')
print(f'Best alpha: {best_alpha:.4g}')
print(f'Baseline MAE: {baseline_mae:.3f}')
print(f'OOF MAE: {oof_mae:.3f}, OOF R2: {oof_r2:.3f}')
print(f'Train MAE: {train_mae:.3f}, Train R2: {train_r2:.3f}')



Joined tracks used by GLM: 46
Usable features retained: 20 / 20
Best alpha: 0.631
Baseline MAE: 1.529
OOF MAE: 1.038, OOF R2: 0.519
Train MAE: 0.846, Train R2: 0.694


In [ ]:
# Predicted vs actual energy diagnostic chart for the regularized GLM (no explicit tempo)
required_glm_vars = ['rows', 'y', 'oof_pred', 'train_pred', 'best_alpha', 'oof_mae', 'train_mae', 'baseline_mae', 'oof_r2', 'train_r2']
missing_glm_vars = [name for name in required_glm_vars if name not in globals()]
if missing_glm_vars:
    raise RuntimeError(f'Run the regularized GLM cell first. Missing variables: {missing_glm_vars}')

actual_energy = np.asarray(y, dtype=np.float64)
oof_energy_pred = np.asarray(oof_pred, dtype=np.float64)
fitted_energy_pred = np.asarray(train_pred, dtype=np.float64)

point_custom = np.array([
    [
        r.get('mix_name', ''),
        r.get('track_number', ''),
        r.get('title', ''),
        r.get('artists', ''),
        r.get('genre', ''),
        r.get('energy', np.nan),
        oof_energy_pred[i],
        fitted_energy_pred[i],
        actual_energy[i] - oof_energy_pred[i],
        actual_energy[i] - fitted_energy_pred[i],
    ]
    for i, r in enumerate(rows)
], dtype=object)

energy_min = float(np.nanmin([np.nanmin(actual_energy), np.nanmin(oof_energy_pred), np.nanmin(fitted_energy_pred), 1.0]))
energy_max = float(np.nanmax([np.nanmax(actual_energy), np.nanmax(oof_energy_pred), np.nanmax(fitted_energy_pred), 9.0]))
pad = max(0.35, 0.06 * (energy_max - energy_min))
axis_range = [energy_min - pad, energy_max + pad]

pred_actual_fig = go.Figure()
pred_actual_fig.add_trace(go.Scatter(
    x=actual_energy,
    y=oof_energy_pred,
    mode='markers',
    name='Out-of-fold prediction',
    customdata=point_custom,
    marker=dict(
        size=12,
        color=actual_energy - oof_energy_pred,
        colorscale='RdBu',
        reversescale=True,
        cmid=0,
        colorbar=dict(title=dict(text='Actual - OOF pred')),
        line=dict(color='white', width=1.2),
        opacity=0.9,
    ),
    hovertemplate=(
        '<b>%{customdata[2]}</b><br>'
        'Artists: %{customdata[3]}<br>'
        'Genre: %{customdata[4]}<br>'
        'Mix: %{customdata[0]} | Track: %{customdata[1]}<br>'
        'Actual energy: %{customdata[5]:.2f}<br>'
        'OOF predicted: %{customdata[6]:.2f}<br>'
        'OOF residual: %{customdata[8]:+.2f}<extra></extra>'
    ),
))
pred_actual_fig.add_trace(go.Scatter(
    x=actual_energy,
    y=fitted_energy_pred,
    mode='markers',
    name='Fitted prediction',
    customdata=point_custom,
    marker=dict(
        size=8,
        color='rgba(23, 32, 42, 0.38)',
        symbol='circle-open',
        line=dict(color='rgba(23, 32, 42, 0.65)', width=1.2),
    ),
    hovertemplate=(
        '<b>%{customdata[2]}</b><br>'
        'Artists: %{customdata[3]}<br>'
        'Actual energy: %{customdata[5]:.2f}<br>'
        'Fitted predicted: %{customdata[7]:.2f}<br>'
        'Fitted residual: %{customdata[9]:+.2f}<extra></extra>'
    ),
))
pred_actual_fig.add_trace(go.Scatter(
    x=axis_range,
    y=axis_range,
    mode='lines',
    name='Perfect prediction',
    line=dict(color='rgba(17, 24, 39, 0.75)', width=2, dash='dash'),
    hoverinfo='skip',
))

# Add +/- 1 energy-point guide bands around the perfect-prediction line.
pred_actual_fig.add_trace(go.Scatter(
    x=axis_range,
    y=[axis_range[0] + 1, axis_range[1] + 1],
    mode='lines',
    name='+/- 1 energy point',
    line=dict(color='rgba(107, 114, 128, 0.35)', width=1, dash='dot'),
    hoverinfo='skip',
))
pred_actual_fig.add_trace(go.Scatter(
    x=axis_range,
    y=[axis_range[0] - 1, axis_range[1] - 1],
    mode='lines',
    name='-1 energy point',
    line=dict(color='rgba(107, 114, 128, 0.35)', width=1, dash='dot'),
    hoverinfo='skip',
    showlegend=False,
))

pred_actual_fig.update_xaxes(
    title='Actual labeled energy',
    range=axis_range,
    dtick=1,
    gridcolor='rgba(133, 146, 166, 0.22)',
)
pred_actual_fig.update_yaxes(
    title='Predicted energy',
    range=axis_range,
    dtick=1,
    gridcolor='rgba(133, 146, 166, 0.22)',
    scaleanchor='x',
    scaleratio=1,
)
pred_actual_fig.update_layout(
    title=(
        'Predicted vs Actual Energy, No Explicit Tempo'
        f'<br><sup>OOF MAE={oof_mae:.3f}, train MAE={train_mae:.3f}, baseline MAE={baseline_mae:.3f}; '
        f'OOF R2={oof_r2:.3f}, train R2={train_r2:.3f}; alpha={best_alpha:.4g}</sup>'
    ),
    width=880,
    height=760,
    template='plotly_white',
    paper_bgcolor='#fbfcfe',
    plot_bgcolor='#fbfcfe',
    margin=dict(l=70, r=40, t=92, b=70),
    legend=dict(orientation='h', yanchor='bottom', y=1.01, xanchor='left', x=0),
    font=dict(size=12, color='#17202a'),
)
pred_actual_fig.show()

prediction_diagnostics = [
    {
        'mix_name': r.get('mix_name', ''),
        'track_number': r.get('track_number', ''),
        'title': r.get('title', ''),
        'artists': r.get('artists', ''),
        'genre': r.get('genre', ''),
        'actual_energy': float(actual_energy[i]),
        'oof_pred': float(oof_energy_pred[i]),
        'fitted_pred': float(fitted_energy_pred[i]),
        'oof_residual': float(actual_energy[i] - oof_energy_pred[i]),
        'fitted_residual': float(actual_energy[i] - fitted_energy_pred[i]),
    }
    for i, r in enumerate(rows)
]

worst_oof_rows = sorted(prediction_diagnostics, key=lambda row: abs(row['oof_residual']), reverse=True)[:8]
worst_oof_fig = go.Figure(data=[go.Table(
    columnwidth=[0.7, 2.4, 1.4, 1.1, 0.8, 0.9, 0.9, 0.9],
    header=dict(
        values=['<b>Mix</b>', '<b>Track</b>', '<b>Artists</b>', '<b>Genre</b>', '<b>Actual</b>', '<b>OOF pred</b>', '<b>Residual</b>', '<b>|Residual|</b>'],
        fill_color='#17202a',
        font=dict(color='white', size=12),
        align=['left', 'left', 'left', 'left', 'right', 'right', 'right', 'right'],
        height=30,
    ),
    cells=dict(
        values=[
            [row['mix_name'] for row in worst_oof_rows],
            [str(row['track_number']) + '. ' + row['title'] for row in worst_oof_rows],
            [row['artists'] for row in worst_oof_rows],
            [row['genre'] for row in worst_oof_rows],
            [f"{row['actual_energy']:.2f}" for row in worst_oof_rows],
            [f"{row['oof_pred']:.2f}" for row in worst_oof_rows],
            [f"{row['oof_residual']:+.2f}" for row in worst_oof_rows],
            [f"{abs(row['oof_residual']):.2f}" for row in worst_oof_rows],
        ],
        fill_color=[['#f8fafc' if i % 2 == 0 else '#eef2f7' for i in range(len(worst_oof_rows))]] * 8,
        font=dict(color='#17202a', size=11),
        align=['left', 'left', 'left', 'left', 'right', 'right', 'right', 'right'],
        height=32,
    ),
)])
worst_oof_fig.update_layout(
    title='Largest Out-of-Fold Prediction Errors',
    width=1100,
    height=380,
    margin=dict(l=20, r=20, t=60, b=20),
    paper_bgcolor='#fbfcfe',
)
worst_oof_fig.show()


In [18]:
# Full Energy Model: explicit tempo + simple + advanced + loudness/RMS audio features
# This mirrors the no-tempo GLM cell above, but keeps all variables suffixed so it does not
# overwrite the no-tempo model used by the export cell.
if 'simple_feature_metrics' not in globals():
    raise RuntimeError('Run cell `aries-centroid-tilt-vs-energy` first to populate `simple_feature_metrics`.')
if 'advanced_feature_metrics' not in globals():
    raise RuntimeError('Run cell `advanced-groove-metrics-vs-energy-by-genre` first to populate `advanced_feature_metrics`.')
if 'loudness_rms_feature_metrics' not in globals():
    raise RuntimeError('Run cell `loudness-rms-vs-labeled-energy-by-genre` first to populate `loudness_rms_feature_metrics`.')

TEMPO_FEATURE_KEYS_FOR_GLM_WITH_TEMPO = ['bpm']
SIMPLE_FEATURE_KEYS_FOR_GLM_WITH_TEMPO = list(SIMPLE_FEATURE_KEYS_FOR_GLM)
ADVANCED_FEATURE_KEYS_FOR_GLM_WITH_TEMPO = list(ADVANCED_FEATURE_KEYS_FOR_GLM)
LOUDNESS_RMS_FEATURE_KEYS_FOR_GLM_WITH_TEMPO = list(globals().get('loudness_rms_feature_keys', [
    'integrated_loudness_lufs',
    'short_term_loudness_mean_lufs',
    'short_term_loudness_std_lu',
    'short_term_loudness_range_lu',
    'rms_mean_db',
    'rms_median_db',
    'rms_std_db',
    'rms_iqr_db',
    'rms_range_db',
    'rms_var',
]))

simple_by_key_with_tempo = {_track_key(m): dict(m) for m in simple_feature_metrics}
advanced_by_key_with_tempo = {_track_key(m): dict(m) for m in advanced_feature_metrics}
loudness_by_key_with_tempo = {_track_key(m): dict(m) for m in loudness_rms_feature_metrics}
common_keys_with_tempo = sorted(
    set(simple_by_key_with_tempo.keys())
    & set(advanced_by_key_with_tempo.keys())
    & set(loudness_by_key_with_tempo.keys())
)

tempo_rows = []
for key in common_keys_with_tempo:
    simple = simple_by_key_with_tempo[key]
    advanced = advanced_by_key_with_tempo[key]
    loudness = loudness_by_key_with_tempo[key]
    energy = _safe_float_or_nan(simple.get('energy', advanced.get('energy', loudness.get('energy'))))
    if not np.isfinite(energy):
        continue

    row = {
        'key': key,
        'mix_name': simple.get('mix_name', advanced.get('mix_name', loudness.get('mix_name', ''))),
        'track_number': simple.get('track_number', advanced.get('track_number', loudness.get('track_number', ''))),
        'title': simple.get('title', advanced.get('title', loudness.get('title', ''))),
        'artists': simple.get('artists', advanced.get('artists', loudness.get('artists', ''))),
        'genre': simple.get('genre', advanced.get('genre', loudness.get('genre', 'unknown'))) or 'unknown',
        'energy': float(energy),
        'bpm': _safe_float_or_nan(simple.get('bpm', advanced.get('bpm', loudness.get('bpm')))),
    }
    for f in SIMPLE_FEATURE_KEYS_FOR_GLM_WITH_TEMPO:
        row[f] = _safe_float_or_nan(simple.get(f))
    for f in ADVANCED_FEATURE_KEYS_FOR_GLM_WITH_TEMPO:
        row[f] = _safe_float_or_nan(advanced.get(f))
    for f in LOUDNESS_RMS_FEATURE_KEYS_FOR_GLM_WITH_TEMPO:
        row[f] = _safe_float_or_nan(loudness.get(f))
    tempo_rows.append(row)

tempo_feature_names = (
    TEMPO_FEATURE_KEYS_FOR_GLM_WITH_TEMPO
    + SIMPLE_FEATURE_KEYS_FOR_GLM_WITH_TEMPO
    + ADVANCED_FEATURE_KEYS_FOR_GLM_WITH_TEMPO
    + LOUDNESS_RMS_FEATURE_KEYS_FOR_GLM_WITH_TEMPO
)

if len(tempo_rows) < 6:
    raise RuntimeError(f'Need at least 6 joined tracks for CV; found {len(tempo_rows)}.')

tempo_X_raw = np.array([[r[f] for f in tempo_feature_names] for r in tempo_rows], dtype=np.float64)
tempo_y = np.array([r['energy'] for r in tempo_rows], dtype=np.float64)

tempo_valid_feature_mask = np.sum(np.isfinite(tempo_X_raw), axis=0) >= GLM_MIN_VALID_FEATURES
tempo_feature_names = [name for name, keep in zip(tempo_feature_names, tempo_valid_feature_mask) if keep]
tempo_X_raw = tempo_X_raw[:, tempo_valid_feature_mask]

if tempo_X_raw.shape[1] == 0:
    raise RuntimeError('No usable features after finite-value filtering.')

tempo_valid_row_mask = np.any(np.isfinite(tempo_X_raw), axis=1) & np.isfinite(tempo_y)
tempo_X_raw = tempo_X_raw[tempo_valid_row_mask]
tempo_y = tempo_y[tempo_valid_row_mask]
tempo_rows = [r for r, keep in zip(tempo_rows, tempo_valid_row_mask) if keep]

if len(tempo_rows) < 6:
    raise RuntimeError(f'Need at least 6 valid tracks after filtering; found {len(tempo_rows)}.')

tempo_base_model = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
    TweedieRegressor(
        power=0,
        link='identity',
        alpha=1.0,
        fit_intercept=True,
        max_iter=10000,
        tol=1e-7,
    ),
)

tempo_n_splits = min(5, len(tempo_rows))
tempo_cv = RepeatedKFold(n_splits=tempo_n_splits, n_repeats=20, random_state=GLM_RANDOM_STATE)

tempo_alpha_scores = []
with warnings.catch_warnings():
    warnings.simplefilter('ignore', ConvergenceWarning)
    for alpha in GLM_ALPHA_GRID:
        fold_mae = []
        for train_idx, test_idx in tempo_cv.split(tempo_X_raw, tempo_y):
            model = clone(tempo_base_model)
            model.set_params(tweedieregressor__alpha=float(alpha))
            model.fit(tempo_X_raw[train_idx], tempo_y[train_idx])
            pred = model.predict(tempo_X_raw[test_idx])
            fold_mae.append(mean_absolute_error(tempo_y[test_idx], pred))
        tempo_alpha_scores.append(float(np.mean(fold_mae)))

tempo_alpha_scores = np.asarray(tempo_alpha_scores, dtype=np.float64)
tempo_best_alpha = float(GLM_ALPHA_GRID[int(np.argmin(tempo_alpha_scores))])

tempo_diagnostic_cv = KFold(n_splits=tempo_n_splits, shuffle=True, random_state=GLM_RANDOM_STATE)
tempo_oof_pred = np.full_like(tempo_y, np.nan, dtype=np.float64)
with warnings.catch_warnings():
    warnings.simplefilter('ignore', ConvergenceWarning)
    for train_idx, test_idx in tempo_diagnostic_cv.split(tempo_X_raw, tempo_y):
        model = clone(tempo_base_model)
        model.set_params(tweedieregressor__alpha=tempo_best_alpha)
        model.fit(tempo_X_raw[train_idx], tempo_y[train_idx])
        tempo_oof_pred[test_idx] = model.predict(tempo_X_raw[test_idx])

tempo_final_model = clone(tempo_base_model)
tempo_final_model.set_params(tweedieregressor__alpha=tempo_best_alpha)
with warnings.catch_warnings():
    warnings.simplefilter('ignore', ConvergenceWarning)
    tempo_final_model.fit(tempo_X_raw, tempo_y)

tempo_coef = tempo_final_model.named_steps['tweedieregressor'].coef_.astype(np.float64)
tempo_intercept = float(tempo_final_model.named_steps['tweedieregressor'].intercept_)
tempo_rank_idx = np.argsort(np.abs(tempo_coef))[::-1]

tempo_baseline_pred = np.full_like(tempo_y, float(np.mean(tempo_y)), dtype=np.float64)
tempo_baseline_mae = float(mean_absolute_error(tempo_y, tempo_baseline_pred))
tempo_oof_mae = float(mean_absolute_error(tempo_y, tempo_oof_pred))
tempo_oof_r2 = float(r2_score(tempo_y, tempo_oof_pred)) if len(np.unique(tempo_y)) > 1 else float('nan')
tempo_train_pred = tempo_final_model.predict(tempo_X_raw)
tempo_train_mae = float(mean_absolute_error(tempo_y, tempo_train_pred))
tempo_train_r2 = float(r2_score(tempo_y, tempo_train_pred)) if len(np.unique(tempo_y)) > 1 else float('nan')

tempo_feature_group_by_name = {
    **{name: 'Tempo metadata' for name in TEMPO_FEATURE_KEYS_FOR_GLM_WITH_TEMPO},
    **{name: 'Simple spectral/groove' for name in SIMPLE_FEATURE_KEYS_FOR_GLM_WITH_TEMPO},
    **{name: 'Advanced rhythm/structure' for name in ADVANCED_FEATURE_KEYS_FOR_GLM_WITH_TEMPO},
    **{name: 'Loudness / RMS' for name in LOUDNESS_RMS_FEATURE_KEYS_FOR_GLM_WITH_TEMPO},
}
tempo_feature_label_by_name = {
    **feature_label_by_name,
    'bpm': 'Tempo BPM tag',
    'integrated_loudness_lufs': 'Integrated loudness',
    'short_term_loudness_mean_lufs': 'Short-term loudness mean',
    'short_term_loudness_std_lu': 'Short-term loudness std',
    'short_term_loudness_range_lu': 'Short-term loudness range',
    'rms_mean_db': 'RMS mean',
    'rms_median_db': 'RMS median',
    'rms_std_db': 'RMS std',
    'rms_iqr_db': 'RMS IQR',
    'rms_range_db': 'RMS range',
    'rms_var': 'RMS variance',
}

glm_with_tempo_coefficient_summary = []
for name, value in zip(tempo_feature_names, tempo_coef):
    group = tempo_feature_group_by_name.get(name, 'Other')
    label = tempo_feature_label_by_name.get(name, name.replace('_', ' ').title())
    glm_with_tempo_coefficient_summary.append({
        'feature': name,
        'label': label,
        'group': group,
        'coefficient': float(value),
        'abs_coefficient': float(abs(value)),
        'direction': 'Positive' if value > 0 else ('Negative' if value < 0 else 'Zero'),
        'finite_count': int(np.sum(np.isfinite(tempo_X_raw[:, tempo_feature_names.index(name)]))),
    })

glm_with_tempo_coefficient_summary = sorted(
    glm_with_tempo_coefficient_summary,
    key=lambda row: (row['abs_coefficient'], row['label'].lower()),
    reverse=True,
)

tempo_plot_rows = list(reversed(glm_with_tempo_coefficient_summary))
tempo_coef_colors = [
    '#0b6b57' if row['coefficient'] >= 0 else '#b7472a'
    for row in tempo_plot_rows
]
tempo_coef_hover = [
    (
        f"<b>{row['label']}</b><br>"
        f"Feature key: {row['feature']}<br>"
        f"Group: {row['group']}<br>"
        f"Standardized coefficient: {row['coefficient']:.4f}<br>"
        f"Finite tracks before imputation: {row['finite_count']}<extra></extra>"
    )
    for row in tempo_plot_rows
]

tempo_coef_fig = go.Figure()
tempo_coef_fig.add_trace(go.Bar(
    x=[row['coefficient'] for row in tempo_plot_rows],
    y=[row['label'] for row in tempo_plot_rows],
    orientation='h',
    marker=dict(
        color=tempo_coef_colors,
        line=dict(color='rgba(20, 28, 38, 0.35)', width=0.8),
    ),
    customdata=np.array([[row['feature'], row['group'], row['finite_count']] for row in tempo_plot_rows], dtype=object),
    hovertemplate=tempo_coef_hover,
    showlegend=False,
))
tempo_coef_fig.add_vline(x=0, line_width=1.5, line_color='rgba(23, 32, 42, 0.7)')

tempo_max_abs_coef = max([row['abs_coefficient'] for row in glm_with_tempo_coefficient_summary] + [0.05])
tempo_coef_fig.update_xaxes(
    title='Standardized coefficient: energy points per +1 SD feature change',
    zeroline=False,
    range=[-tempo_max_abs_coef * 1.18, tempo_max_abs_coef * 1.18],
    gridcolor='rgba(133, 146, 166, 0.22)',
)
tempo_coef_fig.update_yaxes(title='', automargin=True)
tempo_coef_fig.update_layout(
    title=(
        'Full Energy Model Coefficients'
        f'<br><sup>Gaussian GLM / ridge penalty, alpha={tempo_best_alpha:.4g}; '
        f'n={len(tempo_rows)} tracks; OOF MAE={tempo_oof_mae:.3f} vs baseline MAE={tempo_baseline_mae:.3f}; '
        f'OOF R2={tempo_oof_r2:.3f}</sup>'
    ),
    height=max(540, 30 * len(tempo_plot_rows) + 180),
    width=1100,
    margin=dict(l=230, r=40, t=92, b=70),
    template='plotly_white',
    paper_bgcolor='#fbfcfe',
    plot_bgcolor='#fbfcfe',
    font=dict(size=12, color='#17202a'),
    annotations=[
        dict(
            x=0.01,
            y=1.04,
            xref='paper',
            yref='paper',
            text='<span style="color:#0b6b57"><b>Positive</b></span> = higher predicted energy; '
                 '<span style="color:#b7472a"><b>Negative</b></span> = lower predicted energy',
            showarrow=False,
            align='left',
            font=dict(size=12),
        )
    ],
)
tempo_coef_fig.show()

tempo_coef_table_rows = glm_with_tempo_coefficient_summary
tempo_rank_values = list(range(1, len(tempo_coef_table_rows) + 1))
tempo_coef_table_fig = go.Figure(data=[go.Table(
    columnwidth=[0.5, 2.4, 1.35, 1.0, 1.05, 0.75, 1.6],
    header=dict(
        values=[
            '<b>Rank</b>',
            '<b>Feature</b>',
            '<b>Group</b>',
            '<b>Direction</b>',
            '<b>Std. coef</b>',
            '<b>|coef|</b>',
            '<b>Finite tracks</b>',
        ],
        fill_color='#17202a',
        font=dict(color='white', size=12),
        align=['right', 'left', 'left', 'center', 'right', 'right', 'right'],
        height=30,
    ),
    cells=dict(
        values=[
            tempo_rank_values,
            [row['label'] + '<br><span style="color:#6b7280">' + row['feature'] + '</span>' for row in tempo_coef_table_rows],
            [row['group'] for row in tempo_coef_table_rows],
            [row['direction'] for row in tempo_coef_table_rows],
            [f"{row['coefficient']:+.4f}" for row in tempo_coef_table_rows],
            [f"{row['abs_coefficient']:.4f}" for row in tempo_coef_table_rows],
            [row['finite_count'] for row in tempo_coef_table_rows],
        ],
        fill_color=[
            ['#f8fafc' if i % 2 == 0 else '#eef2f7' for i in range(len(tempo_coef_table_rows))],
            ['#f8fafc' if i % 2 == 0 else '#eef2f7' for i in range(len(tempo_coef_table_rows))],
            ['#f8fafc' if i % 2 == 0 else '#eef2f7' for i in range(len(tempo_coef_table_rows))],
            ['#e9f6f2' if row['coefficient'] > 0 else ('#fbeee9' if row['coefficient'] < 0 else '#f2f4f7') for row in tempo_coef_table_rows],
            ['#f8fafc' if i % 2 == 0 else '#eef2f7' for i in range(len(tempo_coef_table_rows))],
            ['#f8fafc' if i % 2 == 0 else '#eef2f7' for i in range(len(tempo_coef_table_rows))],
            ['#f8fafc' if i % 2 == 0 else '#eef2f7' for i in range(len(tempo_coef_table_rows))],
        ],
        font=dict(color='#17202a', size=11),
        align=['right', 'left', 'left', 'center', 'right', 'right', 'right'],
        height=34,
    ),
)])
tempo_coef_table_fig.update_layout(
    title=(
        'Full Energy Model Coefficient Table, Ranked by Absolute Standardized Effect'
        '<br><sup>Full model includes BPM, simple/advanced audio features, and loudness/RMS features. Coefficients are descriptive regularized effects, not causal estimates. '
        'Correlated features can share or swap weight.</sup>'
    ),
    height=max(540, 34 * len(tempo_coef_table_rows) + 130),
    width=1100,
    margin=dict(l=20, r=20, t=82, b=20),
    paper_bgcolor='#fbfcfe',
)
tempo_coef_table_fig.show()

print(f'Joined tracks used by Full Energy Model: {len(tempo_rows)}')
print(f'Usable features retained: {len(tempo_feature_names)} / {len(TEMPO_FEATURE_KEYS_FOR_GLM_WITH_TEMPO) + len(SIMPLE_FEATURE_KEYS_FOR_GLM_WITH_TEMPO) + len(ADVANCED_FEATURE_KEYS_FOR_GLM_WITH_TEMPO) + len(LOUDNESS_RMS_FEATURE_KEYS_FOR_GLM_WITH_TEMPO)}')
print(f'Best alpha for Full Energy Model: {tempo_best_alpha:.4g}')
print(f'Baseline MAE: {tempo_baseline_mae:.3f}')
print(f'OOF MAE for Full Energy Model: {tempo_oof_mae:.3f}, OOF R2: {tempo_oof_r2:.3f}')
print(f'Train MAE for Full Energy Model: {tempo_train_mae:.3f}, Train R2: {tempo_train_r2:.3f}')
if 'oof_mae' in globals() and 'oof_r2' in globals():
    print(f'No-tempo comparison: OOF MAE={oof_mae:.3f}, OOF R2={oof_r2:.3f}')
    print(f'Delta vs no-tempo: MAE {tempo_oof_mae - oof_mae:+.3f}, R2 {tempo_oof_r2 - oof_r2:+.3f}')


Joined tracks used by Full Energy Model: 46
Usable features retained: 31 / 31
Best alpha for Full Energy Model: 1
Baseline MAE: 1.529
OOF MAE for Full Energy Model: 0.985, OOF R2: 0.573
Train MAE for Full Energy Model: 0.765, Train R2: 0.750
No-tempo comparison: OOF MAE=1.038, OOF R2=0.519
Delta vs no-tempo: MAE -0.053, R2 +0.054


In [19]:
# Predicted vs actual energy diagnostic chart for the Full Energy Model
required_tempo_glm_vars = [
    'tempo_rows',
    'tempo_y',
    'tempo_oof_pred',
    'tempo_train_pred',
    'tempo_best_alpha',
    'tempo_oof_mae',
    'tempo_train_mae',
    'tempo_baseline_mae',
    'tempo_oof_r2',
    'tempo_train_r2',
]
missing_tempo_glm_vars = [name for name in required_tempo_glm_vars if name not in globals()]
if missing_tempo_glm_vars:
    raise RuntimeError(f'Run the Full Energy Model cell first. Missing variables: {missing_tempo_glm_vars}')

tempo_actual_energy = np.asarray(tempo_y, dtype=np.float64)
tempo_oof_energy_pred = np.asarray(tempo_oof_pred, dtype=np.float64)
tempo_fitted_energy_pred = np.asarray(tempo_train_pred, dtype=np.float64)

tempo_point_custom = np.array([
    [
        r.get('mix_name', ''),
        r.get('track_number', ''),
        r.get('title', ''),
        r.get('artists', ''),
        r.get('genre', ''),
        r.get('energy', np.nan),
        tempo_oof_energy_pred[i],
        tempo_fitted_energy_pred[i],
        tempo_actual_energy[i] - tempo_oof_energy_pred[i],
        tempo_actual_energy[i] - tempo_fitted_energy_pred[i],
        r.get('bpm', np.nan),
    ]
    for i, r in enumerate(tempo_rows)
], dtype=object)

tempo_energy_min = float(np.nanmin([np.nanmin(tempo_actual_energy), np.nanmin(tempo_oof_energy_pred), np.nanmin(tempo_fitted_energy_pred), 1.0]))
tempo_energy_max = float(np.nanmax([np.nanmax(tempo_actual_energy), np.nanmax(tempo_oof_energy_pred), np.nanmax(tempo_fitted_energy_pred), 9.0]))
tempo_pad = max(0.35, 0.06 * (tempo_energy_max - tempo_energy_min))
tempo_axis_range = [tempo_energy_min - tempo_pad, tempo_energy_max + tempo_pad]

tempo_pred_actual_fig = go.Figure()
tempo_pred_actual_fig.add_trace(go.Scatter(
    x=tempo_actual_energy,
    y=tempo_oof_energy_pred,
    mode='markers',
    name='Out-of-fold prediction',
    customdata=tempo_point_custom,
    marker=dict(
        size=12,
        color=tempo_actual_energy - tempo_oof_energy_pred,
        colorscale='RdBu',
        reversescale=True,
        cmid=0,
        colorbar=dict(title=dict(text='Actual - OOF pred')),
        line=dict(color='white', width=1.2),
        opacity=0.9,
    ),
    hovertemplate=(
        '<b>%{customdata[2]}</b><br>'
        'Artists: %{customdata[3]}<br>'
        'Genre: %{customdata[4]}<br>'
        'Mix: %{customdata[0]} | Track: %{customdata[1]}<br>'
        'BPM tag: %{customdata[10]:.2f}<br>'
        'Actual energy: %{customdata[5]:.2f}<br>'
        'OOF predicted: %{customdata[6]:.2f}<br>'
        'OOF residual: %{customdata[8]:+.2f}<extra></extra>'
    ),
))
tempo_pred_actual_fig.add_trace(go.Scatter(
    x=tempo_actual_energy,
    y=tempo_fitted_energy_pred,
    mode='markers',
    name='Fitted prediction',
    customdata=tempo_point_custom,
    marker=dict(
        size=8,
        color='rgba(23, 32, 42, 0.38)',
        symbol='circle-open',
        line=dict(color='rgba(23, 32, 42, 0.65)', width=1.2),
    ),
    hovertemplate=(
        '<b>%{customdata[2]}</b><br>'
        'Artists: %{customdata[3]}<br>'
        'BPM tag: %{customdata[10]:.2f}<br>'
        'Actual energy: %{customdata[5]:.2f}<br>'
        'Fitted predicted: %{customdata[7]:.2f}<br>'
        'Fitted residual: %{customdata[9]:+.2f}<extra></extra>'
    ),
))
tempo_pred_actual_fig.add_trace(go.Scatter(
    x=tempo_axis_range,
    y=tempo_axis_range,
    mode='lines',
    name='Perfect prediction',
    line=dict(color='rgba(17, 24, 39, 0.75)', width=2, dash='dash'),
    hoverinfo='skip',
))
tempo_pred_actual_fig.add_trace(go.Scatter(
    x=tempo_axis_range,
    y=[tempo_axis_range[0] + 1, tempo_axis_range[1] + 1],
    mode='lines',
    name='+/- 1 energy point',
    line=dict(color='rgba(107, 114, 128, 0.35)', width=1, dash='dot'),
    hoverinfo='skip',
))
tempo_pred_actual_fig.add_trace(go.Scatter(
    x=tempo_axis_range,
    y=[tempo_axis_range[0] - 1, tempo_axis_range[1] - 1],
    mode='lines',
    name='-1 energy point',
    line=dict(color='rgba(107, 114, 128, 0.35)', width=1, dash='dot'),
    hoverinfo='skip',
    showlegend=False,
))

tempo_pred_actual_fig.update_xaxes(
    title='Actual labeled energy',
    range=tempo_axis_range,
    dtick=1,
    gridcolor='rgba(133, 146, 166, 0.22)',
)
tempo_pred_actual_fig.update_yaxes(
    title='Predicted energy',
    range=tempo_axis_range,
    dtick=1,
    gridcolor='rgba(133, 146, 166, 0.22)',
    scaleanchor='x',
    scaleratio=1,
)
tempo_pred_actual_fig.update_layout(
    title=(
        'Predicted vs Actual Energy, Full Energy Model'
        f'<br><sup>OOF MAE={tempo_oof_mae:.3f}, train MAE={tempo_train_mae:.3f}, baseline MAE={tempo_baseline_mae:.3f}; '
        f'OOF R2={tempo_oof_r2:.3f}, train R2={tempo_train_r2:.3f}; alpha={tempo_best_alpha:.4g}</sup>'
    ),
    width=880,
    height=760,
    template='plotly_white',
    paper_bgcolor='#fbfcfe',
    plot_bgcolor='#fbfcfe',
    margin=dict(l=70, r=40, t=92, b=70),
    legend=dict(orientation='h', yanchor='bottom', y=1.01, xanchor='left', x=0),
    font=dict(size=12, color='#17202a'),
)
tempo_pred_actual_fig.show()

tempo_prediction_diagnostics = [
    {
        'mix_name': r.get('mix_name', ''),
        'track_number': r.get('track_number', ''),
        'title': r.get('title', ''),
        'artists': r.get('artists', ''),
        'genre': r.get('genre', ''),
        'bpm': _safe_float_or_nan(r.get('bpm')),
        'actual_energy': float(tempo_actual_energy[i]),
        'oof_pred': float(tempo_oof_energy_pred[i]),
        'fitted_pred': float(tempo_fitted_energy_pred[i]),
        'oof_residual': float(tempo_actual_energy[i] - tempo_oof_energy_pred[i]),
        'fitted_residual': float(tempo_actual_energy[i] - tempo_fitted_energy_pred[i]),
    }
    for i, r in enumerate(tempo_rows)
]

tempo_worst_oof_rows = sorted(tempo_prediction_diagnostics, key=lambda row: abs(row['oof_residual']), reverse=True)[:8]
tempo_worst_oof_fig = go.Figure(data=[go.Table(
    columnwidth=[0.7, 2.4, 1.4, 1.1, 0.7, 0.8, 0.9, 0.9, 0.9],
    header=dict(
        values=['<b>Mix</b>', '<b>Track</b>', '<b>Artists</b>', '<b>Genre</b>', '<b>BPM</b>', '<b>Actual</b>', '<b>OOF pred</b>', '<b>Residual</b>', '<b>|Residual|</b>'],
        fill_color='#17202a',
        font=dict(color='white', size=12),
        align=['left', 'left', 'left', 'left', 'right', 'right', 'right', 'right', 'right'],
        height=30,
    ),
    cells=dict(
        values=[
            [row['mix_name'] for row in tempo_worst_oof_rows],
            [str(row['track_number']) + '. ' + row['title'] for row in tempo_worst_oof_rows],
            [row['artists'] for row in tempo_worst_oof_rows],
            [row['genre'] for row in tempo_worst_oof_rows],
            [f"{row['bpm']:.2f}" if np.isfinite(row['bpm']) else '' for row in tempo_worst_oof_rows],
            [f"{row['actual_energy']:.2f}" for row in tempo_worst_oof_rows],
            [f"{row['oof_pred']:.2f}" for row in tempo_worst_oof_rows],
            [f"{row['oof_residual']:+.2f}" for row in tempo_worst_oof_rows],
            [f"{abs(row['oof_residual']):.2f}" for row in tempo_worst_oof_rows],
        ],
        fill_color=[['#f8fafc' if i % 2 == 0 else '#eef2f7' for i in range(len(tempo_worst_oof_rows))]] * 9,
        font=dict(color='#17202a', size=11),
        align=['left', 'left', 'left', 'left', 'right', 'right', 'right', 'right', 'right'],
        height=32,
    ),
)])
tempo_worst_oof_fig.update_layout(
    title='Largest Out-of-Fold Prediction Errors, Full Energy Model',
    width=1120,
    height=380,
    margin=dict(l=20, r=20, t=60, b=20),
    paper_bgcolor='#fbfcfe',
)
tempo_worst_oof_fig.show()


In [8]:
# Save computed energy features to data/energy_embeddings
from pathlib import Path
from datetime import datetime, timezone
import csv

import numpy as np

ENERGY_EMBEDDINGS_FILENAME = 'aries_ara_energy_features.npz'
ENERGY_MANIFEST_FILENAME = 'aries_ara_energy_features_manifest.csv'

SIMPLE_FEATURE_KEYS_FOR_EXPORT = [
    'spectral_centroid_hz',
    'spectral_tilt_db_per_oct',
    'spectral_rolloff_05_hz',
    'low_band_energy_ratio',
    'low_band_periodicity',
    'dynamic_range_db',
    'crest_factor_db',
    'onset_density',
    'onset_magnitude',
    'rhythmic_entropy',
    'band_flux_low',
    'band_flux_mid',
    'band_flux_high',
]

ADVANCED_FEATURE_KEYS_FOR_EXPORT = [
    'downbeat_strength_ratio',
    'phrase_consistency_4bar',
    'phrase_consistency_1bar',
    'hpss_percussive_ratio',
    'bassline_onset_density',
    'spectral_spread_hz',
    'section_transition_rate',
]


def _project_root_from_cwd() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if (candidate / 'data').exists() and (candidate / 'music').exists():
            return candidate
    return Path.cwd().expanduser().resolve()


def _track_key(row: dict) -> tuple[str, str]:
    return (
        str(row.get('mix_name', '')).strip().lower(),
        str(row.get('mp3_name', '')).strip().lower(),
    )


def _safe_float_or_nan(value) -> float:
    try:
        value_float = float(value)
        return value_float if np.isfinite(value_float) else float('nan')
    except Exception:
        return float('nan')


def _format_csv_value(value):
    try:
        value_float = float(value)
        if not np.isfinite(value_float):
            return ''
        return value_float
    except Exception:
        return value


if 'simple_feature_metrics' not in globals():
    raise RuntimeError('Run cell `aries-centroid-tilt-vs-energy` first to populate `simple_feature_metrics`.')
if 'advanced_feature_metrics' not in globals():
    raise RuntimeError('Run cell `advanced-groove-metrics-vs-energy-by-genre` first to populate `advanced_feature_metrics`.')

# Include GLM fitted/out-of-fold energy predictions if the GLM cell has been run.
glm_by_key = {}
glm_metadata = {
    'glm_available': False,
    'glm_best_alpha': float('nan'),
    'glm_oof_mae': float('nan'),
    'glm_oof_r2': float('nan'),
    'glm_train_mae': float('nan'),
    'glm_train_r2': float('nan'),
}
if all(name in globals() for name in ['rows', 'train_pred', 'oof_pred']):
    try:
        glm_rows = list(globals()['rows'])
        glm_train_pred = np.asarray(globals()['train_pred'], dtype=np.float64).reshape(-1)
        glm_oof_pred = np.asarray(globals()['oof_pred'], dtype=np.float64).reshape(-1)
        if len(glm_rows) == glm_train_pred.size == glm_oof_pred.size:
            for i, glm_row in enumerate(glm_rows):
                key = _track_key(glm_row)
                energy = _safe_float_or_nan(glm_row.get('energy'))
                fitted = float(glm_train_pred[i])
                oof = float(glm_oof_pred[i])
                glm_by_key[key] = {
                    'glm_energy_pred': fitted,
                    'glm_energy_oof_pred': oof,
                    'glm_energy_residual': energy - fitted if np.isfinite(energy) and np.isfinite(fitted) else float('nan'),
                    'glm_energy_oof_residual': energy - oof if np.isfinite(energy) and np.isfinite(oof) else float('nan'),
                }
            glm_metadata = {
                'glm_available': True,
                'glm_best_alpha': _safe_float_or_nan(globals().get('best_alpha')),
                'glm_oof_mae': _safe_float_or_nan(globals().get('oof_mae')),
                'glm_oof_r2': _safe_float_or_nan(globals().get('oof_r2')),
                'glm_train_mae': _safe_float_or_nan(globals().get('train_mae')),
                'glm_train_r2': _safe_float_or_nan(globals().get('train_r2')),
            }
        else:
            print('GLM variables exist but prediction lengths do not match GLM rows; exporting features without GLM values.')
    except Exception as exc:
        print(f'Could not read GLM prediction variables; exporting features without GLM values. Reason: {exc}')
else:
    print('GLM prediction variables not found; exporting features without GLM values. Run the GLM cell first to include them.')

simple_by_key = {_track_key(m): dict(m) for m in simple_feature_metrics}
advanced_by_key = {_track_key(m): dict(m) for m in advanced_feature_metrics}
common_keys = sorted(set(simple_by_key.keys()) & set(advanced_by_key.keys()))

feature_names = SIMPLE_FEATURE_KEYS_FOR_EXPORT + ADVANCED_FEATURE_KEYS_FOR_EXPORT
feature_groups = (
    ['simple'] * len(SIMPLE_FEATURE_KEYS_FOR_EXPORT)
    + ['advanced'] * len(ADVANCED_FEATURE_KEYS_FOR_EXPORT)
)
glm_fields = [
    'glm_energy_pred',
    'glm_energy_oof_pred',
    'glm_energy_residual',
    'glm_energy_oof_residual',
]

export_rows = []
embeddings = []
for key in common_keys:
    simple = simple_by_key[key]
    advanced = advanced_by_key[key]
    energy = _safe_float_or_nan(simple.get('energy', advanced.get('energy')))
    glm_vals = glm_by_key.get(key, {})

    row = {
        'mix_name': str(simple.get('mix_name', advanced.get('mix_name', ''))),
        'track_number': str(simple.get('track_number', advanced.get('track_number', ''))),
        'title': str(simple.get('title', advanced.get('title', ''))),
        'artists': str(simple.get('artists', advanced.get('artists', ''))),
        'genre': str(simple.get('genre', advanced.get('genre', 'unknown')) or 'unknown'),
        'mp3_name': str(simple.get('mp3_name', advanced.get('mp3_name', ''))),
        'energy': energy,
        'bpm': _safe_float_or_nan(simple.get('bpm', advanced.get('bpm'))),
        'glm_energy_pred': _safe_float_or_nan(glm_vals.get('glm_energy_pred')),
        'glm_energy_oof_pred': _safe_float_or_nan(glm_vals.get('glm_energy_oof_pred')),
        'glm_energy_residual': _safe_float_or_nan(glm_vals.get('glm_energy_residual')),
        'glm_energy_oof_residual': _safe_float_or_nan(glm_vals.get('glm_energy_oof_residual')),
    }

    values = []
    for feature in SIMPLE_FEATURE_KEYS_FOR_EXPORT:
        value = _safe_float_or_nan(simple.get(feature))
        row[feature] = value
        values.append(value)
    for feature in ADVANCED_FEATURE_KEYS_FOR_EXPORT:
        value = _safe_float_or_nan(advanced.get(feature))
        row[feature] = value
        values.append(value)

    export_rows.append(row)
    embeddings.append(values)

if not export_rows:
    raise RuntimeError('No overlapping tracks found between simple and advanced energy feature metrics.')

embeddings = np.asarray(embeddings, dtype=np.float32)
project_root = _project_root_from_cwd()
output_dir = project_root / 'data' / 'energy_embeddings'
output_dir.mkdir(parents=True, exist_ok=True)

npz_path = output_dir / ENERGY_EMBEDDINGS_FILENAME
manifest_path = output_dir / ENERGY_MANIFEST_FILENAME
created_utc = datetime.now(tz=timezone.utc).isoformat()

np.savez_compressed(
    npz_path,
    embedding_type=np.array('energy_features', dtype=np.str_),
    created_utc=np.array(created_utc, dtype=np.str_),
    num_tracks=np.array(int(embeddings.shape[0]), dtype=np.int32),
    embedding_dimension=np.array(int(embeddings.shape[1]), dtype=np.int32),
    embeddings=embeddings,
    feature_names=np.asarray(feature_names, dtype=np.str_),
    feature_groups=np.asarray(feature_groups, dtype=np.str_),
    mix_name=np.asarray([r['mix_name'] for r in export_rows], dtype=np.str_),
    track_number=np.asarray([r['track_number'] for r in export_rows], dtype=np.str_),
    title=np.asarray([r['title'] for r in export_rows], dtype=np.str_),
    artists=np.asarray([r['artists'] for r in export_rows], dtype=np.str_),
    genre=np.asarray([r['genre'] for r in export_rows], dtype=np.str_),
    mp3_name=np.asarray([r['mp3_name'] for r in export_rows], dtype=np.str_),
    energy=np.asarray([r['energy'] for r in export_rows], dtype=np.float32),
    bpm=np.asarray([r['bpm'] for r in export_rows], dtype=np.float32),
    glm_available=np.array(bool(glm_metadata['glm_available']), dtype=np.bool_),
    glm_best_alpha=np.array(float(glm_metadata['glm_best_alpha']), dtype=np.float32),
    glm_oof_mae=np.array(float(glm_metadata['glm_oof_mae']), dtype=np.float32),
    glm_oof_r2=np.array(float(glm_metadata['glm_oof_r2']), dtype=np.float32),
    glm_train_mae=np.array(float(glm_metadata['glm_train_mae']), dtype=np.float32),
    glm_train_r2=np.array(float(glm_metadata['glm_train_r2']), dtype=np.float32),
    glm_energy_pred=np.asarray([r['glm_energy_pred'] for r in export_rows], dtype=np.float32),
    glm_energy_oof_pred=np.asarray([r['glm_energy_oof_pred'] for r in export_rows], dtype=np.float32),
    glm_energy_residual=np.asarray([r['glm_energy_residual'] for r in export_rows], dtype=np.float32),
    glm_energy_oof_residual=np.asarray([r['glm_energy_oof_residual'] for r in export_rows], dtype=np.float32),
)

manifest_fields = [
    'mix_name',
    'track_number',
    'title',
    'artists',
    'genre',
    'mp3_name',
    'energy',
    'bpm',
    *glm_fields,
    *feature_names,
]
with manifest_path.open('w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=manifest_fields)
    writer.writeheader()
    for row in export_rows:
        writer.writerow({field: _format_csv_value(row.get(field, '')) for field in manifest_fields})

print(f'Saved energy feature NPZ: {npz_path}')
print(f'Saved energy feature manifest: {manifest_path}')
print(f'Embedding matrix shape: {embeddings.shape}')
print(f'Feature names ({len(feature_names)}): {feature_names}')
print(f'GLM predictions included: {bool(glm_metadata["glm_available"])}')
if glm_metadata['glm_available']:
    print(
        f'GLM best alpha={glm_metadata["glm_best_alpha"]:.4g}, '
        f'OOF MAE={glm_metadata["glm_oof_mae"]:.3f}, '
        f'train MAE={glm_metadata["glm_train_mae"]:.3f}'
    )


Saved energy feature NPZ: /Users/josephdaher/Git Repositories/djprojectexploration/data/energy_embeddings/aries_ara_energy_features.npz
Saved energy feature manifest: /Users/josephdaher/Git Repositories/djprojectexploration/data/energy_embeddings/aries_ara_energy_features_manifest.csv
Embedding matrix shape: (46, 20)
Feature names (20): ['spectral_centroid_hz', 'spectral_tilt_db_per_oct', 'spectral_rolloff_05_hz', 'low_band_energy_ratio', 'low_band_periodicity', 'dynamic_range_db', 'crest_factor_db', 'onset_density', 'onset_magnitude', 'rhythmic_entropy', 'band_flux_low', 'band_flux_mid', 'band_flux_high', 'downbeat_strength_ratio', 'phrase_consistency_4bar', 'phrase_consistency_1bar', 'hpss_percussive_ratio', 'bassline_onset_density', 'spectral_spread_hz', 'section_transition_rate']
GLM predictions included: True
GLM best alpha=0.631, OOF MAE=1.038, train MAE=0.846
